# 03 ML Pipeline: Customer Tribe Discovery

This notebook starts from the prepared parquet outputs created by Notebook 02. It does not repeat raw loading, cleaning, or exploratory data quality work from Notebooks 01 and 02.

The goal is to discover product-first customer tribes from purchase behavior. The client hypothesis is roughly 10-15 tribes, but that range is not used as a modeling constraint. The final recommendation is selected from metrics, stability-ready diagnostics, cluster balance, product/sector lift interpretability, and business usefulness.

## Table of Contents

| Section | What it covers |
|---|---|
| [Stage 0: Load Prepared Data](#stage-0) | Sets the run mode, validates paths, loads prepared transactions, and previews the input data. |
| [Stage 0.5: Expensive Cache Audit](#stage-0-5) | Checks whether Stage 1-6 artifacts can be reused safely from cache. |
| [Stage 1: Basket Construction](#stage-1) | Converts checkout records into basket sentences for product embedding training. |
| [Stage 2: Item2Vec Product Embeddings](#stage-2) | Builds or loads product embeddings from product co-purchase behavior. |
| [Stage 3: Product Embedding Validation](#stage-3) | Reviews product-neighbor diagnostics and embedding quality checks. |
| [Stage 4: Customer Embeddings](#stage-4) | Creates product-first customer embeddings from purchased products and quantities. |
| [Stage 5: Official Feature Set](#stage-5) | Builds the official product/quantity feature set and supporting non-demographic behavior summaries. |
| [Stage 6: Hard UMAP-HDBSCAN Core Tribe Discovery](#stage-6) | Builds the UMAP customer manifold, runs three hard HDBSCAN passes, and exports core-tribe diagnostics. |
| [Stage 6.6: Quality, Stability, Readiness, and Promotion Audit](#stage-6-6) | Tests density quality, stability, confidence, readiness, and promotion blockers before profiling. |
| [Stage 6.7: Remaining-Customer Evidence](#stage-6-7) | Segments customers still unassigned after the hard merge and prepares affinity evidence. |
| [Stage 6.8: Tribe Evidence Assembly](#stage-6-8) | Precomputes all raw-data tribe evidence so Stage 7 can run from cached aggregate artifacts. |
| [Stage 7: Tribe Profiling and Stakeholder Handoff](#stage-7) | Profiles all retained tribes, audits review tribes, analyzes remaining customers, sizes soft audiences, maps relationships, and writes the final handoff. |
| [Stage 8: Dashboard Artifact Pack and Sanity QA](#stage-8) | Builds the final dashboard semantic pack, checks required contracts, and previews every Stage 8 table artifact. |

Tribe discovery is documented as a clear late-stage progression:

| Layer | Pipeline stage | What it proves | Main outputs |
|---|---|---|---|
| 1. Product-only modeling signal | Stages 4-5 | Official clustering uses product identity and quantity, not demographics, spend, or behavior KPIs. | Customer embeddings, feature-set diagnostics |
| 2. Organic core tribes | Stage 6.1-6.8 | Dense product-purchase behavior groups exist without forcing every customer into a tribe, retained clusters are checked before profiling, remaining customers are segmented separately, and all raw evidence is assembled before interpretation. | UMAP representation, hard HDBSCAN assignments, representation/density evidence, stability/readiness report, third-pass HDBSCAN, promotion audit, remaining-customer segments, tribe evidence bundle |
| 3. Tribe profiles | Stage 7 | Promoted and review tribes are profiled from Stage 6.8 aggregate and per-tribe evidence; only promoted tribes enter the final core story. | All-tribe profiles, persona cards, review audit, remaining-customer analysis, relationship atlas, manifest |
| 4. Supporting proof | Stage 7 | Detailed product and comparison artifacts support specific claims without crowding the main read. | Product summaries, tribe comparison, customer metric context |
| 5. Dashboard semantic contract | Stage 8 | The final dashboard can render from structured artifacts only, with no Markdown parsing and no dashboard-side modeling. | Dashboard manifest, artifact inventory, page contracts, deep-dive tables, lookup tables, coverage tables, embeddings, actions, readiness audit |

Read the official result in that order: first whether hard organic core tribes exist, then whether Stage 6.6 stability/readiness supports profiling them, then what Stage 6.7 says about customers still outside the hard tribes, then whether Stage 6.8 assembled the evidence cache cleanly, then whether the Stage 7 evidence storyline makes the official tribes interpretable, distinct, caveated, and profileable, and finally whether Stage 8 packages the complete dashboard contract. Spend and KPIs are interpretation context only; they are never part of the clustering signal.


<a id="stage-0"></a>

## Stage 0: Load Prepared Data

Notebook 03 starts from the prepared transaction parquet created earlier. It validates the fields needed for ML and only merges product metadata if the prepared file does not already contain it. No raw cleaning decisions are made here.


In [ ]:
import os
from pathlib import Path
import sys

from IPython.display import Image, Markdown, display

# Optional notebook override. Set to "dev" or "prod" to force a mode;
# leave as None to honor CARREFOUR_MODE, then the YAML default_mode.
NOTEBOOK_MODE_OVERRIDE = "prod"  # Set to "dev" or "prod" only when you intentionally want to override CARREFOUR_MODE.

for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
    if (candidate / "src").is_dir():
        project_root = candidate
        break
else:
    project_root = Path.cwd().resolve()

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.config import configure_mode, load_config
from src.cache_audit import assert_mode_path_audit
from src.data_loader import load_prepared_transactions, peek
from src.stage_reports import display_stage_report, write_stage_report
from src.utils import set_global_seed

RUN_MODE = (NOTEBOOK_MODE_OVERRIDE or os.environ.get("CARREFOUR_MODE") or load_config().mode).strip().lower()
if RUN_MODE not in {"dev", "prod"}:
    raise ValueError(f"RUN_MODE must be 'dev' or 'prod', got {RUN_MODE!r}")
os.environ["CARREFOUR_MODE"] = RUN_MODE

previous_mode = globals().get("MODE")
CONFIG = configure_mode(RUN_MODE)
MODE = CONFIG.mode
DATA_PROCESSED = CONFIG.data_processed
MODELS = CONFIG.models
OUTPUTS = CONFIG.outputs

if previous_mode and previous_mode != MODE:
    for stale_name in [
        "transactions",
        "basket_path",
        "item2vec_model",
        "product_embeddings_path",
        "embedding_validation_csv",
        "embedding_validation_detail_md",
        "customer_embeddings_path",
        "behavior_path",
        "feature_sets",
        "model_suite",
        "stage6_diagnostics",
        "cluster_stability",
        "stability_table",
        "cluster_readiness_table",
        "stage6_winner_key",
        "stage6_stage1_assignment_path",
        "stage6_stage1_result_path",
        "stage6_stage1_candidate",
        "stage6_stage1_checks_path",
        "stage6_stage1_figure",
        "stage6_stage2_noise_feature_path",
        "stage6_stage2_noise_customers",
        "stage6_stage2_assignment_path",
        "stage6_stage2_result_path",
        "stage6_stage2_candidate",
        "stage6_stage2_checks_path",
        "stage6_stage2_figure",
        "stage6_lift_evidence_path",
        "stage6_lift_evidence",
        "stage6_4_projection_dir",
        "stage6_4_projection_figures",
        "stage6_6_projection_dir",
        "stage6_6_projection_figures",
        "stage6_6_winner_stability",
        "stage6_6_cluster_readiness",
        "stage6_6_cluster_readiness_figure",
        "stage6_7_noise_probe",
        "stage6_7_noise_umap_figure",
        "stage6_7_run_candidate_hdbscan",
        "stage6_7_candidate_probe",
        "stage6_7_diagnostics_path",
        "stage6_7_diagnostics",
        "stage6_7_report",
        "stage6_8_evidence",
        "stage6_8_manifest_path",
        "stage6_8_tribe_evidence_path",
        "stage6_8_product_lifts_path",
        "stage6_8_sector_lifts_path",
        "stage6_8_customer_metric_tests_path",
        "stage6_8_manifest",
        "stage6_8_overview_figure",
        "stage6_8_report",
        "selected_key",
        "selection_evidence_path",
        "selected",
        "selected_profile_path",
        "stage7_artifacts_dir",
        "stage7_figures_dir",
        "stage7_quality",
        "stage7_review_clusters",
        "stage7_final_tribes",
        "stage7_review_candidates",
        "product_summary_paths",
        "tribe_product_tables",
        "tribe_comparison_artifacts",
        "tribe_comparison",
        "customer_metric_tests_path",
        "customer_metric_tests",
        "stage7_significant_customer_metric_tests",
        "stage7_profile_overview",
        "stage7_profile_evidence",
        "stage7_noise_audit",
        "stage7_noise_audit_paths",
        "stage7_noise_summary_row",
        "stage7_storyline_paths",
        "stage7_storyline",
        "stage7_cluster_summary_paths",
        "tribe_vs_population_dashboard",
        "profile_comparison_heatmap",
        "stage7_theme_lift_heatmap",
        "stage7_lift_dir",
        "stage7_final_pack",
        "stage7_final_index",
        "stage7_card_paths",
        "stage7_card_count",
        "stage7_customer_metric_tests_path",
        "stage7_customer_metric_tests",
        "stage7_llm_evidence_path",
        "stage7_gemini_analysis",
        "stage7_gemini_analysis_path",
        "stage7_gemini_status",
        "stage7_gemini_reason",
        "top_lift_paths",
        "profile_export",
        "subsegment_opportunities",
        "subsegment_opportunities_path",
        "customer_subsegment_paths",
        "discovered_term_paths",
        "campaign_signal_paths",
        "profile_report_path",
        "clustering_atlas_paths",
        "llm_prompt_paths",
    ]:
        globals().pop(stale_name, None)

set_global_seed(CONFIG.random_seed)
CONFIG.ensure_directories()
mode_path_audit = assert_mode_path_audit(CONFIG)

transactions = load_prepared_transactions(cfg=CONFIG)
print(f"Run mode: {MODE}")
print(f"Data path: {DATA_PROCESSED}")
print(f"Model path: {MODELS}")
print(f"Output path: {OUTPUTS}")
peek(transactions, 3)

from src.visualization import plot_prepared_data_overview

stage0_figure = plot_prepared_data_overview(transactions, cfg=CONFIG)
stage0_report = write_stage_report(
    "00",
    "Load Prepared Data",
    summary=[
        f"Run mode active: {MODE}",
        "Mode/path audit passed; dev and prod namespaces are not mixed.",
        "Prepared transactions loaded from the configured mode path.",
    ],
    metrics={
        "mode_path_checks": mode_path_audit.height,
        "failed_mode_path_checks": mode_path_audit.filter(mode_path_audit.get_column("status") == "fail").height,
    },
    figures={"Prepared data overview": stage0_figure},
    artifacts={"Prepared transactions": CONFIG.prepared_transactions_path},
    cfg=CONFIG,
)
display_stage_report(stage0_report)
display(mode_path_audit.select(["check", "status", "exists", "reason"]).head(12))
display(Image(filename=str(stage0_figure)))


<a id="stage-0-5"></a>

## Stage 0.5: Expensive Cache Audit

Before expensive stages run, this audit checks whether Stage 1-6 artifacts can be reused. A cache hit requires the file to exist and its metadata hash to match the current inputs/config, so stale artifacts are less likely to slip into the run.


In [ ]:
import importlib
import os
import polars as pl

import src.utils
importlib.reload(src.utils)
import src.cache_audit
importlib.reload(src.cache_audit)
from src.cache_audit import stage_1_6_cache_audit
from src.stage_reports import display_stage_report, write_stage_report
# Optional: if existing artifacts were produced from the current data/config but metadata is missing,
# uncomment the next two lines once to adopt them into the cache manifest.

cache_audit = stage_1_6_cache_audit(CONFIG)
cache_misses = cache_audit.filter(~pl.col("cache_hit"))

stage0_5_report = write_stage_report(
    "00_5",
    "Expensive Cache Audit",
    summary=[
        "Checks whether expensive Stage 1-6 artifacts can be reused safely.",
        "Artifacts with cache_hit=False will be rebuilt when their stage runs.",
    ],
    metrics={
        "audited_artifacts": cache_audit.height,
        "cache_ready_artifacts": cache_audit.filter(pl.col("cache_hit")).height,
        "rebuild_artifacts": cache_misses.height,
    },
    artifacts={"Artifact metadata manifest": CONFIG.outputs / ".artifact_metadata.json"},
    cfg=CONFIG,
)
display_stage_report(stage0_5_report)
cache_display_cols = ["stage", "artifact", "cache_hit", "exists", "reason", "path"]
display(cache_audit.select([c for c in cache_display_cols if c in cache_audit.columns]).head(20))

if cache_misses.height:
    print("Artifacts listed as cache_hit=False will be rebuilt if their stage is run.")
else:
    print("All expensive Stage 1-6 artifacts are cache-ready.")


In [ ]:
# In case of downloaded/ distributes files to make sure we can still run it
from src.cache_audit import adopt_existing_stage_1_6_cache_metadata
display(adopt_existing_stage_1_6_cache_metadata(CONFIG))


<a id="stage-1"></a>

## Stage 1: Basket Construction

Each ticket becomes a basket sentence and each product id becomes a token. The assumption is simple: products bought together reveal shopping missions. Quantity is not repeated as extra tokens unless `baskets.repeat_product_by_quantity` is enabled.


In [ ]:
from IPython.display import Image, display
import polars as pl

from src.basket_builder import (
    basket_summary,
    build_basket_sentences,
    build_basket_staple_diagnostics,
)
from src.stage_reports import display_stage_report, write_stage_report
from src.visualization import (
    plot_basket_staple_diagnostics,
    plot_basket_summary,
)

basket_path = build_basket_sentences(transactions=transactions, cfg=CONFIG)
stage1_basket_summary = basket_summary(basket_path)
stage1_figure = plot_basket_summary(basket_path, cfg=CONFIG)

stage1_diagnostics = build_basket_staple_diagnostics(transactions=transactions, cfg=CONFIG)
stage1_staple_figure = plot_basket_staple_diagnostics(
    stage1_diagnostics["product_diagnostics"],
    stage1_diagnostics["basket_exposure"],
    cfg=CONFIG,
)
stage1_product_diagnostics = pl.scan_parquet(stage1_diagnostics["product_diagnostics"])
stage1_common_products = stage1_product_diagnostics.filter(pl.col("common_product_candidate")).select(pl.len()).collect()[0, 0]
stage1_common_preview_rows = (
    pl.read_csv(stage1_diagnostics["common_products_csv"]).height
    if stage1_diagnostics["common_products_csv"].exists()
    else 0
)
stage1_downsampling_diag_cfg = CONFIG.get("baskets.diagnostics", {}) or {}
stage1_downsampling_plan_path = CONFIG.artifacts / str(stage1_downsampling_diag_cfg.get("output_dir", "stage1")) / str(stage1_downsampling_diag_cfg.get("downsampling_plan_output", "common_product_downsampling_plan.parquet"))
stage1_downsampling_summary_path = CONFIG.artifacts / str(stage1_downsampling_diag_cfg.get("output_dir", "stage1")) / str(stage1_downsampling_diag_cfg.get("downsampling_summary_output", "basket_downsampling_summary.csv"))
stage1_metrics = stage1_basket_summary.row(0, named=True)
stage1_metrics["common_product_candidates"] = stage1_common_products
stage1_metrics["common_product_preview_rows"] = stage1_common_preview_rows
stage1_downsampling_summary = None
if stage1_downsampling_summary_path.exists():
    stage1_downsampling_summary = pl.read_csv(stage1_downsampling_summary_path)
    stage1_metrics.update(
        stage1_downsampling_summary.select(
            [
                "basket_retention_pct",
                "unique_pair_retention_pct",
                "fallback_basket_pct",
                "auto_excluded_products",
                "downsampled_common_candidate_products",
            ]
        ).row(0, named=True)
    )
stage1_report = write_stage_report(
    "01",
    "Basket Sentences and Common-Product Exposure",
    summary=[
        "Basket sentences use product identity; active Stage 1 settings do not repeat products by unidades.",
        "Common-product exposure and downsampling retention are audited before Item2Vec training.",
    ],
    metrics=stage1_metrics,
    figures={
        "Basket token summary": stage1_figure,
        "Common-product exposure": stage1_staple_figure,
    },
    artifacts={
        "Basket sentences": basket_path,
        "Common-product candidates CSV": stage1_diagnostics["common_products_csv"],
        "Product ubiquity diagnostics": stage1_diagnostics["product_diagnostics"],
        "Downsampling plan": stage1_downsampling_plan_path if stage1_downsampling_plan_path.exists() else None,
        "Downsampling summary": stage1_downsampling_summary_path if stage1_downsampling_summary_path.exists() else None,
    },
    cfg=CONFIG,
)
display_stage_report(stage1_report)
display(stage1_basket_summary)
if stage1_downsampling_summary is not None:
    display(stage1_downsampling_summary)
display(Image(filename=str(stage1_figure)))
display(Image(filename=str(stage1_staple_figure)))
print(f"Stage 2 Item2Vec basket path: {basket_path}")


<a id="stage-2"></a>

## Stage 2: Item2Vec Product Embeddings

The Word2Vec model learns product proximity from basket co-occurrence. Products land near each other when they appear in similar baskets, without using demographics, spend, or manual labels.

The training function prints the active hyperparameters and one progress line per epoch. If a cached model already exists, it prints the cache details instead; pass `force=True` to retrain.


In [ ]:
from IPython.display import Image
import polars as pl

from src.item2vec import save_product_embeddings, train_item2vec, write_item2vec_training_diagnostics
from src.stage_reports import display_stage_report, write_stage_report
from src.visualization import plot_product_embedding_diagnostics

item2vec_model = train_item2vec(basket_path, cfg=CONFIG, verbose=True)
stage2_training_diagnostics_path = write_item2vec_training_diagnostics(basket_path, cfg=CONFIG)
stage2_training_diagnostics = pl.read_csv(stage2_training_diagnostics_path)
product_embeddings_path = save_product_embeddings(item2vec_model, cfg=CONFIG)
stage2_figure = plot_product_embedding_diagnostics(product_embeddings_path, cfg=CONFIG)
stage2_schema = pl.read_parquet(product_embeddings_path, n_rows=1).columns
stage2_metrics = {
    "embedded_products": pl.scan_parquet(product_embeddings_path).select(pl.len()).collect()[0, 0],
    "embedding_dimensions": len([c for c in stage2_schema if c.startswith("emb_")]),
    "word2vec_window": CONFIG.get("word2vec.window"),
    "word2vec_min_count": CONFIG.get("word2vec.min_count"),
}
stage2_metrics.update(
    stage2_training_diagnostics.select(
        [
            "baskets",
            "training_baskets",
            "skipped_short_basket_pct",
            "capped_basket_pct",
            "training_token_retention_pct",
            "effective_window",
        ]
    ).row(0, named=True)
)
stage2_report = write_stage_report(
    "02",
    "Product Embedding Training",
    summary=[
        "Item2Vec is trained on Stage 1 product-token baskets after Stage 2 corpus limits.",
        "Product quantities enter the official customer vectors in Stage 4, not as repeated Stage 2 tokens under the active config.",
    ],
    metrics=stage2_metrics,
    figures={"Product embedding diagnostics": stage2_figure},
    artifacts={"Product embeddings": product_embeddings_path, "Training corpus diagnostics": stage2_training_diagnostics_path},
    cfg=CONFIG,
)
display_stage_report(stage2_report)
display(stage2_training_diagnostics)
display(Image(filename=str(stage2_figure)))


<a id="stage-3"></a>

## Stage 3: Product Embedding Validation

Before customer clustering, the nearest-neighbor report checks whether product embeddings capture meaningful substitutes, complements, or shared basket missions. Generic or random-looking neighbors are treated as warning signs.


In [ ]:
from IPython.display import Image
import polars as pl

from src.embedding_validation import product_embedding_guardrail_status, validate_product_embeddings
from src.stage_reports import display_stage_report, write_stage_report
from src.visualization import plot_embedding_validation_quality_extracts, plot_embedding_validation_summary

embedding_validation_csv, embedding_validation_detail_md = validate_product_embeddings(
    product_embeddings_path,
    transactions=transactions,
    cfg=CONFIG,
)
stage3_figure = plot_embedding_validation_summary(embedding_validation_csv, cfg=CONFIG)
stage3_extract_figures = {}
if CONFIG.get("embedding_validation.write_extract_figures", False):
    stage3_extract_figures = plot_embedding_validation_quality_extracts(embedding_validation_csv, cfg=CONFIG)
stage3_guardrail_status = product_embedding_guardrail_status(embedding_validation_csv, cfg=CONFIG)
stage3_figures = {"Embedding validation summary": stage3_figure}
stage3_figures.update({f"Quality extract: {name}": path for name, path in stage3_extract_figures.items()})
stage3_issues = stage3_guardrail_status["issues"] or ["None"]
stage3_report = write_stage_report(
    "03",
    "Product Embedding Validation",
    summary=[
        f"Guardrail status: {stage3_guardrail_status['status']}",
        stage3_guardrail_status["summary"],
        "Issues: " + "; ".join(stage3_issues[:4]),
    ],
    metrics={
        "validated_products": pl.read_csv(embedding_validation_csv).height,
        "guardrail_issue_count": 0 if stage3_guardrail_status["issues"] is None else len(stage3_guardrail_status["issues"]),
    },
    figures=stage3_figures,
    artifacts={
        "Validation CSV": embedding_validation_csv,
        "Hubness CSV": stage3_guardrail_status["hubness_csv"],
    },
    cfg=CONFIG,
)
display_stage_report(stage3_report)
display(Image(filename=str(stage3_figure)))
for figure_path in stage3_extract_figures.values():
    display(Image(filename=str(figure_path)))
stage3_guardrail_status


<a id="stage-4"></a>

## Stage 4: Customer Embeddings

Each customer becomes one vector: a weighted average of the products they bought. Weighting uses quantity, recency decay, and repeat-basket frequency, so clustering is driven by product choice and purchase intensity rather than spend. IDF variants stay sandbox-only unless explicitly promoted in config.


In [ ]:
from IPython.display import Image, display
import polars as pl

from src.customer_embeddings import build_customer_embeddings
from src.stage_reports import display_stage_report, write_stage_report
from src.visualization import plot_customer_embedding_diagnostics

customer_embeddings_path = build_customer_embeddings(
    product_embeddings_path,
    transactions=transactions,
    normalize_vectors=CONFIG.get("customer_embeddings.normalize_vectors", False),
    cfg=CONFIG,
)
stage4_figure = plot_customer_embedding_diagnostics(customer_embeddings_path, cfg=CONFIG)

stage4_diag_cfg = CONFIG.get("customer_embeddings.diagnostics", {}) or {}
stage4_weight_summary_path = (
    CONFIG.artifacts
    / str(stage4_diag_cfg.get("output_dir", "stage4"))
    / f"{stage4_diag_cfg.get('output_prefix', 'customer_embedding')}_weight_diagnostics.csv"
)
stage4_display_cols = [
    "weight_strategy",
    "quantity_transform",
    "common_product_weight_share_pct",
    "mean_customer_common_product_weight_share_pct",
    "p95_customer_common_product_weight_share_pct",
    "top_product_weight_share_pct",
    "top_10_product_weight_share_pct",
    "mean_customer_top_product_weight_share_pct",
    "stage4_gate_status",
    "stage4_gate_issues",
    "recency_weighting_enabled",
    "recency_reference_date",
    "recency_half_life_days",
    "mean_recency_multiplier",
    "frequency_weighting_enabled",
    "frequency_transform",
    "mean_frequency_multiplier",
    "p95_customer_product_basket_count",
    "line_coverage_pct",
    "unit_coverage_pct",
    "product_coverage_pct",
    "customer_coverage_pct",
    "zero_embedded_customer_pct",
]
stage4_weight_summary = None
stage4_report_metrics = {
    "embedded_customers": pl.scan_parquet(customer_embeddings_path).select(pl.len()).collect()[0, 0]
}
if stage4_weight_summary_path.exists():
    stage4_weight_summary = pl.read_csv(stage4_weight_summary_path)
    available_stage4_cols = [c for c in stage4_display_cols if c in stage4_weight_summary.columns]
    if available_stage4_cols:
        stage4_report_metrics.update(stage4_weight_summary.select(available_stage4_cols).row(0, named=True))
stage4_report = write_stage_report(
    "04",
    "Customer Embeddings Coverage and Dominance Gates",
    summary=[
        "Customer vectors are weighted aggregations of purchased-product embeddings.",
        "Coverage gates check how much transaction signal survives product embedding coverage.",
        "Dominance gates check whether common products overwhelm customer vectors.",
    ],
    metrics=stage4_report_metrics,
    figures={"Customer embedding diagnostics": stage4_figure},
    artifacts={
        "Customer embeddings": customer_embeddings_path,
        "Weight diagnostics": stage4_weight_summary_path if stage4_weight_summary_path.exists() else None,
    },
    cfg=CONFIG,
    max_metric_rows=32,
)
display_stage_report(stage4_report)
display(Image(filename=str(stage4_figure)))
if stage4_weight_summary is not None:
    display(stage4_weight_summary.select([c for c in stage4_display_cols if c in stage4_weight_summary.columns]))


<a id="stage-5"></a>

## Stage 5: Official Feature Set

This stage locks the official clustering input. Behavioral features and spend KPIs are saved separately for profiling, but they do not enter the clustering feature set. The modeling boundary is product/quantity customer embeddings only.


In [ ]:
from IPython.display import Image
import polars as pl

from src.feature_engineering import build_behavioral_features, build_feature_set, build_feature_set_diagnostics, build_product_exposure_features
from src.stage_reports import display_stage_report, write_stage_report
from src.visualization import plot_behavioral_feature_summary, plot_feature_set_summary

behavior_path = build_behavioral_features(transactions=transactions, cfg=CONFIG)
selection_feature_set = CONFIG.get("modeling.feature_set_for_selection", "embeddings_only")
build_product_exposure_challenger = bool(CONFIG.get("product_exposure_features.enabled", False)) or selection_feature_set == "embeddings_product_exposure"
product_exposure_path = None
embeddings_only_feature_set = build_feature_set(customer_embeddings_path, variant="embeddings_only", cfg=CONFIG)
feature_sets = {"embeddings_only": embeddings_only_feature_set}
if build_product_exposure_challenger:
    product_exposure_path = build_product_exposure_features(transactions=transactions, cfg=CONFIG)
    feature_sets["embeddings_product_exposure"] = build_feature_set(
        customer_embeddings_path,
        product_exposure_path=product_exposure_path,
        variant="embeddings_product_exposure",
        cfg=CONFIG,
    )
stage5_behavior_figure = plot_behavioral_feature_summary(behavior_path, cfg=CONFIG)
stage5_feature_set_figure = plot_feature_set_summary(feature_sets, cfg=CONFIG)
stage5_feature_diagnostics_path = build_feature_set_diagnostics(feature_sets, cfg=CONFIG)
stage5_feature_diagnostics = pl.read_csv(stage5_feature_diagnostics_path)
stage5_diagnostic_display_cols = [
    "feature_set_name",
    "is_selection_feature_set",
    "rows",
    "feature_count",
    "finite_pct",
    "null_pct",
    "zero_variance_feature_count",
    "customer_alignment_status",
    "missing_vs_baseline_customers",
    "extra_vs_baseline_customers",
    "neighbor_overlap_vs_baseline_pct",
    "selection_warning",
]
stage5_report = write_stage_report(
    "05",
    "Official Product/Quantity Feature Set",
    summary=[
        "Official tribe selection uses embeddings_only to keep clustering product-first.",
        "Behavioral features are prepared separately for post-clustering interpretation.",
        "Product-exposure challenger features are opt-in and skipped by default to keep prod runs lean.",
    ],
    metrics={
        "feature_sets_built": len(feature_sets),
        "selection_feature_set": CONFIG.get("modeling.feature_set_for_selection", "embeddings_only"),
        "feature_diagnostics_rows": stage5_feature_diagnostics.height,
        "product_exposure_challenger_enabled": build_product_exposure_challenger,
    },
    figures={
        "Behavioral feature summary": stage5_behavior_figure,
        "Feature set summary": stage5_feature_set_figure,
    },
    artifacts={
        **{"Behavioral features": behavior_path, "Feature diagnostics": stage5_feature_diagnostics_path},
        **({"Product exposure features": product_exposure_path} if product_exposure_path is not None else {}),
        **feature_sets,
    },
    cfg=CONFIG,
)
display_stage_report(stage5_report)
display(Image(filename=str(stage5_behavior_figure)))
display(Image(filename=str(stage5_feature_set_figure)))
display(stage5_feature_diagnostics.select([c for c in stage5_diagnostic_display_cols if c in stage5_feature_diagnostics.columns]))


<a id="stage-6"></a>

## Stage 6: Customer Tribe Discovery (UMAP + HDBSCAN)

Three-pass density clustering on product embeddings. Customers who share purchasing patterns cluster together organically — no demographic inputs, no predefined labels.

- **Passes 1–3**: Each pass runs HDBSCAN on customers still unassigned from the prior pass, recovering progressively smaller niche groups.
- **Lift filter**: Only clusters where members buy certain products ≥ 2× more than average are kept as tribes.
- **Coverage rescue (6.5a)**: Remaining noise customers are soft-assigned to the nearest valid tribe centroid within the q75 intra-cluster distance bound. Rescue is for activation sizing only — product lift profiling uses core members exclusively.
- **Quality audit (6.6)**: Each cluster earns a `strong`, `usable`, or `review` readiness label before any interpretation begins.

A tribe is only claimed when density evidence, product lift, and stability all point to the same conclusion.

In [ ]:
import polars as pl
from pathlib import Path
from IPython.display import Image, Markdown, display

from src.model_selection import (
    build_candidate_model_diagnostics,
    build_official_umap_core_representation,
    build_stage6_hdbscan_diagnostics,
    build_stage6_representation_cluster_diagnostics,
    build_stage6_umap_diagnostics,
    apply_noise_rescue_soft_assignment,
    merge_official_three_stage_hdbscan_lift_core,
    model_suite_from_single_candidate,
    run_official_three_stage_hdbscan_first_pass,
    run_official_three_stage_hdbscan_second_pass,
    run_official_three_stage_hdbscan_third_pass,
)
from src.stage_reports import display_stage_report, write_stage_report
from src.visualization import (
    plot_noise_rescue_tribe_breakdown,
    plot_stage6_hdbscan_assignment_map,
    plot_stage6_noise_rescue_provenance,
    plot_stage6_quality_evidence,
    plot_stage6_umap_representation,
)

selection_feature_set = CONFIG.get("modeling.feature_set_for_selection", "embeddings_only")
if "feature_sets" not in globals():
    feature_set_outputs = CONFIG.get("feature_sets.outputs", {})
    selection_feature_filename = feature_set_outputs.get(selection_feature_set, f"feature_set_{selection_feature_set}.parquet")
    feature_sets = {
        selection_feature_set: CONFIG.outputs / "features" / selection_feature_filename,
    }
    missing_feature_sets = [path for path in feature_sets.values() if not path.exists()]
    if missing_feature_sets:
        raise FileNotFoundError(
            "Stage 6 needs the Stage 5 feature-set parquet files. "
            f"Missing: {missing_feature_sets}. Run Stage 5 first."
        )

allowed_product_feature_sets = {"embeddings_only", "embeddings_product_exposure"}
if selection_feature_set not in allowed_product_feature_sets:
    raise ValueError(
        "Official Stage 6 core tribe discovery only allows product-derived feature sets: "
        f"{sorted(allowed_product_feature_sets)}. Do not use behavior, spend, or demographic feature sets here."
    )
stage6_feature_path = feature_sets[selection_feature_set]
stage6_feature_path


### Stage 6.1: UMAP Representation Check

Build the PCA-to-UMAP customer manifold used for density clustering. PCA reduces noise before UMAP; the checks verify row alignment, dimensions, nulls, finite values, and basic neighborhood quality before HDBSCAN runs.


In [ ]:
stage6_umap_path = build_official_umap_core_representation(stage6_feature_path, cfg=CONFIG)
stage6_umap_checks_path = build_stage6_umap_diagnostics(stage6_feature_path, stage6_umap_path, cfg=CONFIG)
stage6_umap_checks = pl.read_csv(stage6_umap_checks_path)
stage6_pca_summary_path = CONFIG.artifacts / "stage6" / "stage6_1_pca_for_umap_summary.csv"
stage6_pca_summary = pl.read_csv(stage6_pca_summary_path)
stage6_pca_overview = stage6_pca_summary.select(
    [
        "purpose",
        "dimension_reduction",
        "requested_component_count",
        "retained_component_count",
        pl.col("retained_variance_pct").round(2),
        pl.col("pc1_variance_pct").round(2),
        pl.col("pc5_cumulative_variance_pct").round(2),
        pl.col("pc10_cumulative_variance_pct").round(2),
    ]
)
stage6_umap_figure = plot_stage6_umap_representation(stage6_umap_path, cfg=CONFIG)

display(stage6_pca_overview)
display(stage6_umap_checks)
display(Image(filename=str(stage6_umap_figure)))
if stage6_umap_checks[0, "check_status"] != "pass":
    raise ValueError(f"Stage 6.1 UMAP checks failed: {stage6_umap_checks[0, 'check_issues']}")


### Stage 6.2: First Hard HDBSCAN Pass

Run hard HDBSCAN on the full UMAP representation. This finds the first core tribes and leaves uncertain customers as first-pass noise for separate review.


In [ ]:
stage6_stage1_assignment_path, stage6_stage1_result_path, stage6_stage1_candidate = run_official_three_stage_hdbscan_first_pass(
    stage6_umap_path,
    cfg=CONFIG,
)
stage6_stage1_checks_path = build_stage6_hdbscan_diagnostics(
    stage6_umap_path,
    stage6_stage1_assignment_path,
    stage6_stage1_result_path,
    output_path=CONFIG.artifacts / "stage6" / "stage6_2_first_hdbscan_checks.csv",
    stage_name="6.2_first_hdbscan",
    cfg=CONFIG,
)
stage6_stage1_checks = pl.read_csv(stage6_stage1_checks_path)
stage6_stage1_assignment_source_table = (
    pl.read_parquet(stage6_stage1_assignment_path)
    .group_by("assignment_source")
    .agg(pl.len().alias("customers"))
    .sort("customers", descending=True)
)
stage6_stage1_figure = plot_stage6_hdbscan_assignment_map(
    stage6_umap_path,
    stage6_stage1_assignment_path,
    output_path=CONFIG.figures / "stage_06_2_first_hdbscan_assignment_map.png",
    title="Stage 6.2 First HDBSCAN Core Tribes on UMAP",
    cfg=CONFIG,
)

display(stage6_stage1_checks)
display(stage6_stage1_assignment_source_table)
display(Image(filename=str(stage6_stage1_figure)))
blocking_status_col = "blocking_check_status" if "blocking_check_status" in stage6_stage1_checks.columns else "check_status"
blocking_issues_col = "blocking_check_issues" if "blocking_check_issues" in stage6_stage1_checks.columns else "check_issues"
if stage6_stage1_checks[0, blocking_status_col] != "pass":
    raise ValueError(f"Stage 6.2 first HDBSCAN structural checks failed: {stage6_stage1_checks[0, blocking_issues_col]}")


### Stage 6.3: Second HDBSCAN Pass Over First-Pass Noise

Run a stricter HDBSCAN only on first-pass noise. This can recover dense niche groups, but customers remain unassigned unless a real density pocket is found.


In [ ]:
(
    stage6_stage2_noise_feature_path,
    stage6_stage2_noise_customers,
    stage6_stage2_assignment_path,
    stage6_stage2_result_path,
    stage6_stage2_candidate,
) = run_official_three_stage_hdbscan_second_pass(
    stage6_umap_path,
    stage1_assignment_path=stage6_stage1_assignment_path,
    cfg=CONFIG,
)

if stage6_stage2_assignment_path is not None and stage6_stage2_result_path is not None:
    stage6_stage2_checks_path = build_stage6_hdbscan_diagnostics(
        stage6_stage2_noise_feature_path,
        stage6_stage2_assignment_path,
        stage6_stage2_result_path,
        output_path=CONFIG.artifacts / "stage6" / "stage6_3_second_noise_hdbscan_checks.csv",
        stage_name="6.3_second_noise_hdbscan",
        cfg=CONFIG,
    )
    stage6_stage2_checks = pl.read_csv(stage6_stage2_checks_path)
    stage6_stage2_assignment_source_table = (
        pl.read_parquet(stage6_stage2_assignment_path)
        .group_by("assignment_source")
        .agg(pl.len().alias("customers"))
        .sort("customers", descending=True)
    )
    stage6_stage2_figure = plot_stage6_hdbscan_assignment_map(
        stage6_stage2_noise_feature_path,
        stage6_stage2_assignment_path,
        output_path=CONFIG.figures / "stage_06_3_second_noise_hdbscan_assignment_map.png",
        title="Stage 6.3 Second HDBSCAN Pass on First-Pass Noise",
        cfg=CONFIG,
    )
    display(stage6_stage2_checks)
    display(stage6_stage2_assignment_source_table)
    display(Image(filename=str(stage6_stage2_figure)))
    blocking_status_col = "blocking_check_status" if "blocking_check_status" in stage6_stage2_checks.columns else "check_status"
    blocking_issues_col = "blocking_check_issues" if "blocking_check_issues" in stage6_stage2_checks.columns else "check_issues"
    if stage6_stage2_checks[0, blocking_status_col] != "pass":
        raise ValueError(f"Stage 6.3 second HDBSCAN structural checks failed: {stage6_stage2_checks[0, blocking_issues_col]}")
else:
    stage6_stage2_checks_path = None
    stage6_stage2_figure = None
    stage6_stage2_assignment_source_table = pl.DataFrame()
    stage6_stage2_checks = pl.DataFrame([
        {
            "stage": "6.3_second_noise_hdbscan",
            "check_status": "skipped",
            "check_issues": "first-pass noise below second-pass min_cluster_size",
            "noise_customers": stage6_stage2_noise_customers,
        }
    ])
    display(Markdown(
        f"Stage 6.3 skipped: first-pass noise customers ({stage6_stage2_noise_customers:,}) "
        "are below the configured second-pass min_cluster_size."
    ))
    display(stage6_stage2_checks)


### Stage 6.4: Third Hard HDBSCAN Pass Over Remaining Noise

Run one final hard HDBSCAN pass on customers still unassigned after Stage 6.3. This is a last chance to find dense remaining pockets; the later merge and product-lift filter still decide whether those pockets are retained.


In [ ]:
(
    stage6_stage3_noise_feature_path,
    stage6_stage3_noise_customers,
    stage6_stage3_assignment_path,
    stage6_stage3_result_path,
    stage6_stage3_candidate,
) = run_official_three_stage_hdbscan_third_pass(
    stage6_umap_path,
    stage1_assignment_path=stage6_stage1_assignment_path,
    stage2_assignment_path=stage6_stage2_assignment_path,
    cfg=CONFIG,
)

if stage6_stage3_assignment_path is not None and stage6_stage3_result_path is not None:
    stage6_stage3_checks_path = build_stage6_hdbscan_diagnostics(
        stage6_stage3_noise_feature_path,
        stage6_stage3_assignment_path,
        stage6_stage3_result_path,
        output_path=CONFIG.artifacts / "stage6" / "stage6_4_third_remaining_noise_hdbscan_checks.csv",
        stage_name="6.4_third_remaining_noise_hdbscan",
        cfg=CONFIG,
    )
    stage6_stage3_checks = pl.read_csv(stage6_stage3_checks_path)
    stage6_stage3_assignment_source_table = (
        pl.read_parquet(stage6_stage3_assignment_path)
        .group_by("assignment_source")
        .agg(pl.len().alias("customers"))
        .sort("customers", descending=True)
    )
    stage6_stage3_figure = plot_stage6_hdbscan_assignment_map(
        stage6_stage3_noise_feature_path,
        stage6_stage3_assignment_path,
        output_path=CONFIG.figures / "stage_06_4_third_remaining_noise_hdbscan_assignment_map.png",
        title="Stage 6.4 Third HDBSCAN Pass on Remaining Noise",
        cfg=CONFIG,
    )
    display(stage6_stage3_checks)
    display(stage6_stage3_assignment_source_table)
    display(Image(filename=str(stage6_stage3_figure)))
    blocking_status_col = "blocking_check_status" if "blocking_check_status" in stage6_stage3_checks.columns else "check_status"
    blocking_issues_col = "blocking_check_issues" if "blocking_check_issues" in stage6_stage3_checks.columns else "check_issues"
    if stage6_stage3_checks[0, blocking_status_col] != "pass":
        raise ValueError(f"Stage 6.4 third HDBSCAN structural checks failed: {stage6_stage3_checks[0, blocking_issues_col]}")
else:
    stage6_stage3_checks_path = None
    stage6_stage3_figure = None
    stage6_stage3_assignment_source_table = pl.DataFrame()
    stage6_stage3_checks = pl.DataFrame([
        {
            "stage": "6.4_third_remaining_noise_hdbscan",
            "check_status": "skipped",
            "check_issues": "remaining second-pass noise below third-pass minimum",
            "noise_customers": stage6_stage3_noise_customers,
        }
    ])
    display(Markdown(
        f"Stage 6.4 skipped: remaining second-pass noise customers ({stage6_stage3_noise_customers:,}) "
        "are below the configured third-pass minimum."
    ))
    display(stage6_stage3_checks)


### Stage 6.5: Merge Three Passes and Apply Product-Lift Filter

Merge retained clusters from the three HDBSCAN passes, then keep only clusters with enough strong significant product-lift evidence. Unsupported candidate clusters and residual noise stay unassigned.


In [ ]:
stage6_assignment_path, stage6_result_path, stage6_core_candidate = merge_official_three_stage_hdbscan_lift_core(
    stage6_feature_path,
    stage6_umap_path,
    stage1_assignment_path=stage6_stage1_assignment_path,
    stage1_results_path=stage6_stage1_result_path,
    stage2_assignment_path=stage6_stage2_assignment_path,
    stage2_results_path=stage6_stage2_result_path,
    stage2_noise_feature_path=stage6_stage2_noise_feature_path,
    stage2_noise_customers=stage6_stage2_noise_customers,
    stage3_assignment_path=stage6_stage3_assignment_path,
    stage3_results_path=stage6_stage3_result_path,
    stage3_noise_feature_path=stage6_stage3_noise_feature_path,
    stage3_noise_customers=stage6_stage3_noise_customers,
    cfg=CONFIG,
)
stage6_hdbscan_checks_path = build_stage6_hdbscan_diagnostics(
    stage6_umap_path,
    stage6_assignment_path,
    stage6_result_path,
    output_path=CONFIG.artifacts / "stage6" / "stage6_5_merged_three_stage_hdbscan_checks.csv",
    stage_name="6.5_merged_three_stage_lift_core",
    cfg=CONFIG,
)
stage6_hdbscan_checks = pl.read_csv(stage6_hdbscan_checks_path)
stage6_assignment_source_table = (
    pl.read_parquet(stage6_assignment_path)
    .group_by("assignment_source")
    .agg(pl.len().alias("customers"))
    .sort("customers", descending=True)
)
stage6_lift_evidence_path = stage6_core_candidate.get("lift_filter_evidence_csv_path")
stage6_lift_evidence = pl.read_csv(stage6_lift_evidence_path) if stage6_lift_evidence_path else pl.DataFrame()
stage6_assignment_figure = plot_stage6_hdbscan_assignment_map(
    stage6_umap_path,
    stage6_assignment_path,
    output_path=CONFIG.figures / "stage_06_5_merged_three_stage_hdbscan_assignment_map.png",
    title="Stage 6.5 Merged Three-Stage HDBSCAN Lift-Core Tribes on UMAP",
    cfg=CONFIG,
)

display(stage6_hdbscan_checks)
display(stage6_assignment_source_table)
# Per-tribe breakdown: tribe name, HDBSCAN pass, hard-core customers, core share
_pass_src = pl.DataFrame({
    "assignment_source": [
        "three_stage_hdbscan_stage1_core",
        "three_stage_hdbscan_stage2_noise_core",
        "three_stage_hdbscan_stage3_remaining_noise_core",
    ],
    "hdbscan_pass": ["Pass 1", "Pass 2", "Pass 3"],
})
_n_total = pl.read_parquet(stage6_assignment_path).height
_tribe_promo_path = CONFIG.artifacts / "stage7" / f"stage7_1_tribe_promotion_report_{CONFIG.mode}.csv"
stage6_tribe_pass_table = (
    pl.read_parquet(stage6_assignment_path)
    .filter(pl.col("tribe_id") >= 0)
    .group_by(["tribe_id", "assignment_source"])
    .agg(pl.len().alias("customers"))
    .join(_pass_src, on="assignment_source", how="left")
    .with_columns((pl.col("customers") / _n_total * 100).round(1).alias("core_share_pct"))
    .sort("tribe_id")
)
if _tribe_promo_path.exists():
    _names = pl.read_csv(_tribe_promo_path).select(["tribe_id", "business_name"])
    stage6_tribe_pass_table = (
        stage6_tribe_pass_table
        .join(_names, on="tribe_id", how="left")
        .select(["tribe_id", "business_name", "hdbscan_pass", "customers", "core_share_pct"])
    )
display(stage6_tribe_pass_table)

if not stage6_lift_evidence.is_empty():
    display(stage6_lift_evidence)
display(Image(filename=str(stage6_assignment_figure)))
blocking_status_col = "blocking_check_status" if "blocking_check_status" in stage6_hdbscan_checks.columns else "check_status"
blocking_issues_col = "blocking_check_issues" if "blocking_check_issues" in stage6_hdbscan_checks.columns else "check_issues"
if stage6_hdbscan_checks[0, blocking_status_col] != "pass":
    raise ValueError(f"Stage 6.5 merged HDBSCAN structural checks failed: {stage6_hdbscan_checks[0, blocking_issues_col]}")


### Stage 6.5a: Post-Hoc Coverage Rescue (Activation Layer)

After the three hard HDBSCAN passes and the product-lift filter, customers still at `tribe_id = -1` are assigned to the nearest valid tribe centroid in 20-D UMAP space. A customer is rescued only if its centroid distance falls within the q75 intra-cluster threshold.

**This is an activation layer, not tribe discovery.** Hard HDBSCAN assignments are unchanged. Rescued customers are used for:
- Dashboard coverage counts (total tribe reach)
- Campaign audience sizing (Stage 7.7)

They are **excluded** from product lift profiling (Stage 6.8) to avoid diluting tribe distinctiveness.

In [ ]:
stage6_activation_assignment_path, stage6_noise_rescue_stats = apply_noise_rescue_soft_assignment(
    stage6_assignment_path,
    stage6_umap_path,
    cfg=CONFIG,
)
stage6_activation_assignment_path = Path(stage6_activation_assignment_path)
stage6_noise_rescue_assignment_path = Path(
    stage6_noise_rescue_stats.get("noise_rescue_path", stage6_activation_assignment_path)
)
stage6_core_profile_assignment_path = Path(
    stage6_noise_rescue_stats.get("core_only_path", stage6_assignment_path)
)
stage6_noise_rescue_summary = pl.DataFrame([stage6_noise_rescue_stats])
stage6_activation_source_table = (
    pl.read_parquet(stage6_activation_assignment_path)
    .group_by("assignment_source")
    .agg(pl.len().alias("customers"))
    .sort("customers", descending=True)
)
stage6_noise_rescue_provenance_figure = plot_stage6_noise_rescue_provenance(
    stage6_umap_path,
    stage6_activation_assignment_path,
    output_path=CONFIG.figures / "stage_06_5a_noise_rescue_provenance.png",
    title="Stage 6.5a Coverage Rescue - Activation Layer Only",
    cfg=CONFIG,
)
stage6_noise_rescue_breakdown_figure = plot_noise_rescue_tribe_breakdown(
    stage6_activation_assignment_path,
    output_path=CONFIG.figures / "stage_06_5a_noise_rescue_tribe_breakdown.png",
    cfg=CONFIG,
)

display(stage6_noise_rescue_summary)
display(stage6_activation_source_table)
# Per-tribe rescue breakdown: name, hard-core, rescued, total, population share
_n_pop = pl.read_parquet(stage6_activation_assignment_path).height
_tribe_promo_path = CONFIG.artifacts / "stage7" / f"stage7_1_tribe_promotion_report_{CONFIG.mode}.csv"
_core_by_tribe = (
    pl.read_parquet(stage6_assignment_path)
    .filter(pl.col("tribe_id") >= 0)
    .group_by("tribe_id")
    .agg(pl.len().alias("hard_core_customers"))
)
_rescue_by_tribe = (
    pl.read_parquet(stage6_activation_assignment_path)
    .filter(pl.col("assignment_source") == "noise_rescue_nearest_centroid_q75")
    .group_by("tribe_id")
    .agg(pl.len().alias("rescued_customers"))
)
stage6_rescue_tribe_table = (
    _core_by_tribe
    .join(_rescue_by_tribe, on="tribe_id", how="left")
    .with_columns(pl.col("rescued_customers").fill_null(0))
    .with_columns((pl.col("hard_core_customers") + pl.col("rescued_customers")).alias("total_customers"))
    .with_columns((pl.col("total_customers") / _n_pop * 100).round(1).alias("population_share_pct"))
    .sort("tribe_id")
)
if _tribe_promo_path.exists():
    _names = pl.read_csv(_tribe_promo_path).select(["tribe_id", "business_name"])
    stage6_rescue_tribe_table = (
        stage6_rescue_tribe_table
        .join(_names, on="tribe_id", how="left")
        .select(["tribe_id", "business_name", "hard_core_customers", "rescued_customers", "total_customers", "population_share_pct"])
    )
display(stage6_rescue_tribe_table)

display(Image(filename=str(stage6_noise_rescue_provenance_figure)))
display(Image(filename=str(stage6_noise_rescue_breakdown_figure)))


### Stage 6 Quality Evidence: Representation and Density

After the merged hard assignment exists, write compact evidence about representation quality, density validity, noise, balance, and core-only separation. This is model-quality context before the cluster-level readiness audit.


In [ ]:
model_suite = model_suite_from_single_candidate(
    stage6_assignment_path,
    stage6_result_path,
    stage6_core_candidate,
    umap_path=stage6_umap_path,
)
stage6_diagnostics = build_candidate_model_diagnostics(model_suite, cfg=CONFIG)
stage6_quality_path = build_stage6_representation_cluster_diagnostics(
    stage6_feature_path,
    stage6_umap_path,
    stage6_assignment_path,
    stage6_result_path,
    output_path=CONFIG.artifacts / "stage6" / "stage6_6_representation_cluster_quality.csv",
    cfg=CONFIG,
)
stage6_quality = pl.read_csv(stage6_quality_path)
stage6_figure = plot_stage6_quality_evidence(
    stage6_quality_path,
    output_path=CONFIG.figures / "stage_06_6_quality_evidence.png",
    title="Stage 6.6 Representation and Assignment Evidence",
    rescue_stats=stage6_noise_rescue_stats if "stage6_noise_rescue_stats" in globals() else None,
    cfg=CONFIG,
)
stage6_ranked = pl.read_parquet(stage6_diagnostics["parquet"]).sort("stage6_rank")
stage6_quality_display_cols = [
    "aligned_rows",
    "source_feature_count",
    "umap_component_count",
    "umap_quality_sample_size",
    "umap_neighbor_k",
    "umap_trustworthiness",
    "umap_mean_knn_overlap_pct",
    "umap_distance_spearman",
    "cluster_count",
    "noise_pct",
    "core_coverage_pct",
    "avg_assignment_confidence",
]
stage6_quality_table = stage6_quality.select([c for c in stage6_quality_display_cols if c in stage6_quality.columns])
stage6_display_cols = [
    "stage6_rank",
    "candidate_id",
    "model_name",
    "algorithm_name",
    "assignment_policy",
    "model_variant",
    "cluster_count",
    "noise_pct",
    "core_coverage_pct",
    "lift_supported_cluster_count",
    "lift_rejected_cluster_count",
    "passes_quality_gate",
]
stage6_table = stage6_ranked.select([c for c in stage6_display_cols if c in stage6_ranked.columns])
stage6_best = stage6_ranked.row(0, named=True)
stage6_quality_best = stage6_quality.row(0, named=True)
_rescue_stats = stage6_noise_rescue_stats if "stage6_noise_rescue_stats" in globals() else {}
_rescued_customers = int((_rescue_stats or {}).get("rescued_customers", 0))
_still_noise = int((_rescue_stats or {}).get("still_noise_customers", 0))
_aligned_rows = int(stage6_quality_best.get("aligned_rows") or 0)
_post_rescue_noise_pct = round(100.0 * _still_noise / max(_aligned_rows, 1), 2) if _aligned_rows else None

stage6_report_metrics = [
    ("pca_dimension_reduction", stage6_pca_summary[0, "dimension_reduction"], "pre-UMAP PCA reduction"),
    ("pca_retained_variance_pct", stage6_pca_summary[0, "retained_variance_pct"], "standardized variance retained before UMAP"),
    ("first_pass_cluster_count", stage6_stage1_candidate.get("cluster_count"), "diagnostic first pass"),
    ("first_pass_noise_pct", stage6_stage1_candidate.get("noise_pct"), "noise inspected in Stage 6.3"),
    ("second_pass_cluster_count", stage6_stage2_candidate.get("cluster_count"), "recovered from first-pass noise"),
    ("second_pass_noise_customers", stage6_stage2_noise_customers, "remaining candidates for second pass"),
    ("third_pass_cluster_count", stage6_stage3_candidate.get("cluster_count"), "recovered from remaining second-pass noise"),
    ("third_pass_noise_customers", stage6_stage3_noise_customers, "remaining candidates for third pass"),
    ("unfiltered_cluster_count", stage6_best.get("unfiltered_cluster_count"), "before product-lift filtering"),
    ("lift_supported_cluster_count", stage6_best.get("lift_supported_cluster_count"), "official retained clusters"),
    ("lift_rejected_cluster_count", stage6_best.get("lift_rejected_cluster_count"), "clusters rejected for weak product evidence"),
    ("cluster_count", stage6_quality_best.get("cluster_count"), "10-15 client hypothesis; not a hard constraint"),
    ("noise_pct", stage6_quality_best.get("noise_pct"), "<= 60% quality gate; <= 40% strong (raw HDBSCAN output, before centroid rescue)"),
    ("rescued_customers", _rescued_customers, "centroid rescue: noise customers reassigned to nearest tribe"),
    ("post_rescue_noise_pct", _post_rescue_noise_pct, "% remaining unassigned after centroid rescue; target < 10%"),
    ("core_coverage_pct", stage6_quality_best.get("core_coverage_pct"), ">= 40% useful; >= 60% strong"),
    ("quality_gate_status", stage6_hdbscan_checks[0, "quality_gate_status"] if "quality_gate_status" in stage6_hdbscan_checks.columns else stage6_best.get("passes_quality_gate"), "pass"),
    ("quality_gate_issues", stage6_hdbscan_checks[0, "quality_gate_issues"] if "quality_gate_issues" in stage6_hdbscan_checks.columns else stage6_best.get("quality_gate_reason"), "pass / None"),
    ("umap_trustworthiness", stage6_quality_best.get("umap_trustworthiness"), ">= 0.95 strong; >= 0.90 usable"),
    ("umap_mean_knn_overlap_pct", stage6_quality_best.get("umap_mean_knn_overlap_pct"), ">= 25% useful; >= 15% review"),
    ("umap_distance_spearman", stage6_quality_best.get("umap_distance_spearman"), "supporting context"),
    ("avg_assignment_confidence", stage6_quality_best.get("avg_assignment_confidence"), ">= 0.40 stronger; >= 0.30 review"),
    ("centroid_rescue_strategy", stage6_noise_rescue_stats.get("strategy") if "stage6_noise_rescue_stats" in globals() else None, "post-hoc activation layer"),
    ("centroid_rescued_customers", stage6_noise_rescue_stats.get("rescued_customers") if "stage6_noise_rescue_stats" in globals() else None, "assigned for activation coverage only"),
    ("still_unassigned_after_rescue", stage6_noise_rescue_stats.get("still_noise_customers") if "stage6_noise_rescue_stats" in globals() else None, "remain outside tribes"),
    ("assignment_policy", stage6_best.get("assignment_policy"), "hard_three_stage_hdbscan_lift_core_noise_retained"),
    ("soft_assigned_pct", stage6_best.get("soft_assigned_pct"), "0% for official hard-core recipe"),
]
stage6_report = write_stage_report(
    "06",
    "Hard Three-Stage UMAP-HDBSCAN Core Tribe Discovery",
    summary=[
        f"Selection feature set: {selection_feature_set} (official signal is product identity, quantity, and optional product-derived exposure; no spend or demographic inputs).",
        f"Stage 6.1 PCA pre-reduces {stage6_pca_summary[0, 'dimension_reduction']} dimensions and retains {stage6_pca_summary[0, 'retained_variance_pct']:.1f}% standardized variance before UMAP.",
        "Stage 6.1 builds the UMAP customer manifold; Stage 6.2 runs first-pass hard HDBSCAN; Stage 6.3 reruns stricter HDBSCAN over first-pass noise.",
        "Stage 6.4 runs the third pass; Stage 6.5 merges all three passes and keeps only clusters with strong significant product-lift evidence.",
        "Stage 6.6 reports stakeholder-safe representation quality, transparent noise handling, assignment confidence, and readiness evidence.",
        "Stage 6.5a adds a separate centroid-rescue activation layer; Stage 6.8 profiling remains core-only to protect product lift evidence.",
        f"Official core model: {stage6_best['model_name']} / {stage6_best['model_variant']}",
    ],
    metrics=stage6_report_metrics,
    figures={
        "Stage 6.1 UMAP representation": stage6_umap_figure,
        "Stage 6.2 first HDBSCAN assignment map": stage6_stage1_figure,
        "Stage 6.3 second-pass noise HDBSCAN assignment map": stage6_stage2_figure,
        "Stage 6.5 merged lift-core assignment map": stage6_assignment_figure,
        "Stage 6.5a rescue provenance map": stage6_noise_rescue_provenance_figure if "stage6_noise_rescue_provenance_figure" in globals() else None,
        "Stage 6.5a hard-vs-rescue breakdown": stage6_noise_rescue_breakdown_figure if "stage6_noise_rescue_breakdown_figure" in globals() else None,
        "Stage 6.6 representation and assignment evidence": stage6_figure,
    },
    artifacts={
        "Stage 6.1 PCA summary": stage6_pca_summary_path,
        "UMAP representation": model_suite.get("umap_path"),
        "Stage 6.1 UMAP checks": stage6_umap_checks_path,
        "Stage 6.2 first-pass assignments": stage6_stage1_assignment_path,
        "Stage 6.2 first-pass checks": stage6_stage1_checks_path,
        "Stage 6.3 noise feature subset": stage6_stage2_noise_feature_path,
        "Stage 6.3 second-pass assignments": stage6_stage2_assignment_path,
        "Stage 6.3 second-pass checks": stage6_stage2_checks_path,
        "Stage 6.5 merged lift-core assignments": stage6_assignment_path,
        "Stage 6.5a activation/rescued assignments": stage6_activation_assignment_path if "stage6_activation_assignment_path" in globals() else None,
        "Stage 6.5a core-only profiling assignments": stage6_core_profile_assignment_path if "stage6_core_profile_assignment_path" in globals() else stage6_assignment_path,
        "Stage 6.5 merged checks": stage6_hdbscan_checks_path,
        "Stage 6.5 lift filter evidence": stage6_lift_evidence_path,
        "Stage 6.6 representation and cluster quality CSV": stage6_quality_path,
        "Diagnostics parquet": stage6_diagnostics["parquet"],
        "Diagnostics summary CSV": stage6_diagnostics["summary_csv"],
    },
    cfg=CONFIG,
    max_metric_rows=30,
)
display_stage_report(stage6_report)
display(Image(filename=str(stage6_figure)))
display(stage6_quality_table)
display(stage6_table.head(12))
display(stage6_assignment_source_table)


<a id="stage-6-6"></a>

## Stage 6.6: Tribe Quality Audit

Each cluster passes stability and confidence checks before it is called a tribe. The three readiness labels are:

| Label | Meaning |
|-------|---------|
| **✓ Strong** | Jitter recovery ≥ 80%, confident assignments. Fully validated — use freely. |
| **~ Usable** | Promoted but below the strong threshold. Reliable, but note the caveat. |
| **? Review** | Failed a quality gate (too few customers, unstable geometry, low confidence). Not a final tribe — treated as a hypothesis. |

**Jitter recovery** is a stability test: add small noise to customer vectors, re-assign to nearest centroid, measure how often the original tribe is recovered. Low recovery means the tribe's shape is fragile or sits close to another cluster boundary.

In [ ]:
from IPython.display import Image, display
import polars as pl

from src.cluster_validation import build_cluster_promotion_audit, build_cluster_validity_stability_report
from src.stage_reports import display_stage_report, write_stage_report
from src.visualization import plot_stage66_winner_umap_tribe_breakdown, plot_stage6_cluster_readiness

cluster_stability = build_cluster_validity_stability_report(
    model_suite,
    feature_sets[selection_feature_set],
    output_path=CONFIG.artifacts / "stage6" / "stage6_6_cluster_stability_readiness.parquet",
    cfg=CONFIG,
)

stability_table = pl.read_csv(cluster_stability["summary_csv"]).select([
    "candidate_id",
    "cluster_count",
    "noise_pct",
    "cluster_size_cv",
    "jitter_ari_mean",
    "jitter_ari_std",
    "jitter_label_recovery_accuracy_mean",
    "validity_note",
])
stage6_winner_key = pl.read_parquet(stage6_diagnostics["parquet"]).sort("stage6_rank")[0, "candidate_id"]
stage6_6_winner_stability = stability_table.filter(pl.col("candidate_id") == stage6_winner_key)
cluster_readiness_table = pl.read_csv(cluster_stability["cluster_summary_csv"])
stage6_6_cluster_readiness = cluster_readiness_table.filter(pl.col("candidate_id") == stage6_winner_key)

stage6_6_promotion_audit_paths = build_cluster_promotion_audit(
    stage6_core_candidate.get("unfiltered_assignment_path"),
    stage6_core_candidate.get("lift_filter_evidence_path"),
    cluster_stability["cluster_summary_csv"],
    output_csv=CONFIG.artifacts / "stage6" / f"stage6_6_cluster_promotion_audit_{CONFIG.mode}.csv",
    output_md=CONFIG.artifacts / "stage6" / f"stage6_6_cluster_promotion_audit_{CONFIG.mode}.md",
    cfg=CONFIG,
)
stage6_6_promotion_audit = pl.read_csv(stage6_6_promotion_audit_paths["csv"])

stage6_6_projection_dir = CONFIG.figures
_rescue_assignment = CONFIG.model_selection_cache / "cluster_assignments_model_e_three_stage_hdbscan_lift_core_noise_rescued.parquet"
_winner_assignment = _rescue_assignment if _rescue_assignment.exists() else model_suite["assignment_paths"][stage6_winner_key]
stage6_6_tribe_breakdown_figure = plot_stage66_winner_umap_tribe_breakdown(
    stage6_umap_path,
    _winner_assignment,
    output_path=stage6_6_projection_dir / f"stage6_6_winner_projection_umap_tribe_breakdown_{CONFIG.mode}.png",
    title="Stage 6.6 Winner — 22 Tribes on UMAP & Core/Rescue Composition",
    cfg=CONFIG,
)
stage6_6_cluster_readiness_figure = plot_stage6_cluster_readiness(
    cluster_stability["cluster_summary_csv"],
    output_path=CONFIG.figures / "stage_06_6_cluster_readiness.png",
    title="Stage 6.6 Cluster Readiness Before Profiling",
    cfg=CONFIG,
)
stage6_6_metrics_row = stage6_6_winner_stability.row(0, named=True)
stage6_6_report_metrics = [
    ("cluster_count", stage6_6_metrics_row.get("cluster_count"), "10-15 client hypothesis; not a hard constraint"),
    ("noise_pct", stage6_6_metrics_row.get("noise_pct"), "<= 60% quality gate; <= 40% strong"),
    ("cluster_size_cv", stage6_6_metrics_row.get("cluster_size_cv"), "<= 1.5 gate; lower is more balanced"),
    ("jitter_ari_mean", stage6_6_metrics_row.get("jitter_ari_mean"), ">= 0.80 strong; >= 0.60 usable"),
    ("jitter_label_recovery_accuracy_mean", stage6_6_metrics_row.get("jitter_label_recovery_accuracy_mean"), ">= 0.80 strong; < 0.60 review blocker"),
    ("clusters_marked_review", stage6_6_cluster_readiness.filter(pl.col("profile_readiness") == "review").height, "0 ideal; review before profiling if > 0"),
    ("promotion_audit_review_not_promoted", stage6_6_promotion_audit.filter(pl.col("promotion_status") == "review_not_promoted").height, "retained but not promoted"),
    ("promotion_audit_rejected_lift_filter", stage6_6_promotion_audit.filter(pl.col("promotion_status") == "rejected_lift_filter").height, "candidate clusters rejected for product evidence"),
    ("clusters_marked_usable_or_strong", stage6_6_cluster_readiness.filter(pl.col("profile_readiness").is_in(["usable", "strong"])).height, "all retained clusters"),
]
stage6_6_report = write_stage_report(
    "06_6",
    "Quality, Stability, Readiness, and Promotion Audit",
    summary=[
        f"Stage 6 winner checked: {stage6_winner_key}",
        "Stage 6.6 focuses on the retained and rejected candidate clusters: quality, stability, confidence, size, readiness, and explicit promotion blockers.",
        "This is the technical and evidence gate before Stage 7 business interpretation.",
    ],
    metrics=stage6_6_report_metrics,
    figures={
        "Cluster readiness": stage6_6_cluster_readiness_figure,
        "UMAP tribe breakdown": stage6_6_tribe_breakdown_figure,
    },
    artifacts={
        "Stability/readiness parquet": cluster_stability["parquet"],
        "Stability/readiness summary CSV": cluster_stability["summary_csv"],
        "Cluster readiness parquet": cluster_stability["cluster_parquet"],
        "Cluster readiness summary CSV": cluster_stability["cluster_summary_csv"],
        "Cluster promotion audit CSV": stage6_6_promotion_audit_paths["csv"],
        "Cluster promotion audit MD": stage6_6_promotion_audit_paths["markdown"],
        "Winner UMAP tribe breakdown": stage6_6_tribe_breakdown_figure,
    },
    cfg=CONFIG,
    max_metric_rows=16,
)
display_stage_report(stage6_6_report)
display(stability_table.head(20))
display(stage6_6_winner_stability)
display(stage6_6_promotion_audit)

display(stage6_6_cluster_readiness.select([
    "tribe_id",
    "customers",
    "customer_share_pct",
    "mean_assignment_confidence",
    "p10_assignment_confidence",
    "jitter_label_recovery_accuracy_mean",
    "profile_readiness",
    "readiness_issues",
]).sort("tribe_id"))
display(Image(filename=str(stage6_6_cluster_readiness_figure)))
display(Image(filename=str(stage6_6_tribe_breakdown_figure)))


<a id="stage-6-7"></a>

## Stage 6.7: Remaining-Customer Evidence

Stage 6.7 analyzes customers **still unassigned after both the hard HDBSCAN passes (Stages 6.2–6.5) and the centroid rescue (Stage 6.5a)**. This is the genuine residual — shoppers whose purchase pattern did not land close enough to any tribe even after the rescue pass.

These are **not** the 54% noise figure from Stage 6.6 (which was pre-rescue). By this point, rescue has already absorbed a large portion of that noise pool, so the remaining group here is materially smaller.

Descriptive segments are business-readable characterizations — not hidden tribes. Soft-affinity scores flag which of these customers sits nearest to a specific tribe, for optional campaign targeting only.

In [ ]:
from IPython.display import Image, Markdown, display
import polars as pl

from src.profiling import write_remaining_customer_segment_artifacts
from src.stage_reports import display_stage_report, write_stage_report
from src.visualization import plot_stage67_remaining_segment_bar, plot_stage67_segment_definitions

stage6_7_remaining_paths = write_remaining_customer_segment_artifacts(
    stage6_activation_assignment_path if "stage6_activation_assignment_path" in globals() else stage6_assignment_path,
    behavior_path=behavior_path if "behavior_path" in globals() else None,
    output_csv=CONFIG.artifacts / "stage6" / f"stage6_7_remaining_customer_segments_{CONFIG.mode}.csv",
    output_md=CONFIG.artifacts / "stage6" / f"stage6_7_remaining_customer_segments_{CONFIG.mode}.md",
    output_html=CONFIG.artifacts / "stage6" / f"stage6_7_remaining_customer_segments_{CONFIG.mode}.html",
    affinity_output_parquet=CONFIG.artifacts / "stage6" / f"stage6_7_remaining_customer_affinity_{CONFIG.mode}.parquet",
    cfg=CONFIG,
)
stage6_7_remaining_segments = pl.read_csv(stage6_7_remaining_paths["csv"])
stage6_7_affinity = pl.read_parquet(stage6_7_remaining_paths["affinity_parquet"])
stage6_7_affinity_summary = (
    stage6_7_affinity.group_by("affinity_confidence_band")
    .agg(pl.len().alias("customers"))
    .sort("customers", descending=True)
    if not stage6_7_affinity.is_empty()
    else pl.DataFrame()
)

stage6_7_report_metrics = [
    ("remaining_customer_segments", stage6_7_remaining_segments.height, "business-readable descriptive segments"),
    ("remaining_customers", int(stage6_7_remaining_segments["customer_count"].sum()) if not stage6_7_remaining_segments.is_empty() else 0, "still unassigned after centroid rescue"),
    ("affinity_rows", stage6_7_affinity.height, "remaining customers scored against hard-tribe centroids"),
    ("high_confidence_affinity_rows", stage6_7_affinity.filter(pl.col("affinity_confidence_band") == "high").height if not stage6_7_affinity.is_empty() else 0, "eligible for later soft-audience review"),
]
stage6_7_report = write_stage_report(
    "06_7",
    "Remaining-Customer Evidence",
    summary=[
        "Stage 6.7 analyzes customers still outside tribes after centroid rescue.",
        "Segments are descriptive business groups, not hidden hard tribes.",
        "Soft affinity is campaign-use evidence only and does not alter official membership.",
    ],
    metrics=stage6_7_report_metrics,
    artifacts={
        "Remaining customer segments CSV": stage6_7_remaining_paths["csv"],
        "Remaining customer segments MD": stage6_7_remaining_paths["markdown"],
        "Remaining customer segments HTML": stage6_7_remaining_paths["html"],
        "Remaining customer affinity parquet": stage6_7_remaining_paths["affinity_parquet"],
    },
    cfg=CONFIG,
    max_metric_rows=16,
)
display_stage_report(stage6_7_report)
display(stage6_7_remaining_segments)

stage6_7_segment_bar_figure = plot_stage67_remaining_segment_bar(
    stage6_7_remaining_segments,
    output_path=CONFIG.figures / f"stage6_7_remaining_segment_bar_{CONFIG.mode}.png",
    cfg=CONFIG,
)
stage6_7_segment_def_figure = plot_stage67_segment_definitions(
    stage6_7_remaining_segments,
    output_path=CONFIG.figures / f"stage6_7_segment_definitions_{CONFIG.mode}.png",
    cfg=CONFIG,
)
display(Markdown("#### Customer count per remaining-customer segment"))
display(Image(filename=str(stage6_7_segment_bar_figure)))
display(Markdown("#### What each segment means"))
display(Image(filename=str(stage6_7_segment_def_figure)))

if not stage6_7_affinity_summary.is_empty():
    display(Markdown("### Remaining-Customer Soft Affinity Bands"))
    display(stage6_7_affinity_summary)


<a id="stage-6-8"></a>

## Stage 6.8: Evidence Assembly

Locks tribe membership and pre-computes all profiling data: product lifts, sector lifts, behavioral KPIs, spend metrics, and loyalty segments — one evidence record per tribe.

Profiling uses **core HDBSCAN members only** (centroid-rescued customers are excluded). Rescued customers re-enter for activation audience and coverage counts in Stage 7.7.

In [ ]:
import json
import polars as pl
from IPython.display import Image, display

from src.profiling import build_stage68_tribe_evidence
from src.stage_reports import display_stage_report, write_stage_report
from src.visualization import plot_stage68_evidence_overview

cluster_readiness_source = cluster_stability.get("cluster_summary_csv") if "cluster_stability" in globals() else None
stage6_profile_assignment_path = stage6_core_profile_assignment_path if "stage6_core_profile_assignment_path" in globals() else stage6_assignment_path
_rescue_path = stage6_activation_assignment_path if "stage6_activation_assignment_path" in globals() else None
stage6_8_evidence = build_stage68_tribe_evidence(
    stage6_profile_assignment_path,
    rescue_assignments_path=_rescue_path,
    cluster_readiness_path=cluster_readiness_source,
    behavior_path=behavior_path if "behavior_path" in globals() else None,
    cfg=CONFIG,
)
stage6_8_manifest_path = stage6_8_evidence["manifest_json"]
stage6_8_tribe_evidence_path = stage6_8_evidence["tribe_evidence_path"]
stage6_8_product_lifts_path = stage6_8_evidence["product_lifts_path"]
stage6_8_sector_lifts_path = stage6_8_evidence["sector_lifts_path"]
stage6_8_customer_metric_tests_path = stage6_8_evidence["customer_metric_tests_csv"]
stage6_8_manifest = stage6_8_evidence["manifest"]
stage6_8_counts = stage6_8_manifest.get("row_counts", {})
stage6_8_readiness_summary = stage6_8_manifest.get("readiness_summary", {})
stage6_8_overview_figure = plot_stage68_evidence_overview(
    stage6_8_tribe_evidence_path,
    noise_vs_core_path=stage6_8_evidence.get("noise_vs_core_customer_metrics_csv"),
    output_path=CONFIG.figures / f"stage6_8_tribe_evidence_overview_{CONFIG.mode}.png",
    cfg=CONFIG,
)
stage6_8_report = write_stage_report(
    "06_8",
    "Tribe Evidence Assembly",
    summary=[
        "Stage 6.8 precomputes product lifts, sector lifts, co-purchase, behavioral, temporal, loyalty, ANOVA, and analyst exports from hard HDBSCAN core members only.",
        "The retained evidence count is allowed to differ from the final promoted profile count; Stage 7 profiles only the Stage 6.6 promoted readiness statuses carried here.",
        "Stage 7 consumes the saved core-only evidence parquet, Stage 6.8 manifest, and per-tribe exports; the rescued assignment is used only for audience coverage/activation.",
        f"Evidence root: {stage6_8_manifest_path.parent}",
    ],
    metrics=[
        ("tribe_evidence_rows", stage6_8_counts.get("tribe_evidence"), "one row per retained Stage 6 tribe before readiness filtering"),
        ("stage6_6_promoted_tribes", stage6_8_readiness_summary.get("promoted_tribes"), "Stage 7 card/index candidates"),
        ("stage6_6_review_tribes", stage6_8_readiness_summary.get("review_tribes"), "held for review; no Stage 7 card/index row"),
        ("stage6_6_missing_readiness", stage6_8_readiness_summary.get("missing_readiness_tribes"), "0 required for a coherent Stage 6.8-to-Stage 7 handoff"),
        ("product_lift_rows", stage6_8_counts.get("product_lifts"), "full tribe x product evidence table"),
        ("sector_lift_rows", stage6_8_counts.get("sector_lifts"), "full tribe x sector evidence table"),
        ("customer_metric_tests", stage6_8_counts.get("customer_metric_tests"), "precomputed ANOVA rows"),
        ("transaction_export_files", len(stage6_8_manifest.get("transaction_exports", [])), "one parquet per tribe"),
        ("customer_export_files", len(stage6_8_manifest.get("customer_exports", [])), "one customer KPI parquet per tribe"),
    ],
    figures={
        "Stage 6.8 evidence overview": stage6_8_overview_figure,
    },
    artifacts={
        "Stage 6.8 manifest": stage6_8_manifest_path,
        "Core-only profiling assignment": stage6_profile_assignment_path,
        "Activation/rescued assignment": stage6_activation_assignment_path if "stage6_activation_assignment_path" in globals() else None,
        "Tribe evidence parquet": stage6_8_tribe_evidence_path,
        "Product lifts parquet": stage6_8_product_lifts_path,
        "Sector lifts parquet": stage6_8_sector_lifts_path,
        "Customer metric ANOVA CSV": stage6_8_customer_metric_tests_path,
        "Transaction exports": stage6_8_evidence.get("transaction_export_dir"),
        "Customer summary exports": stage6_8_evidence.get("customer_export_dir"),
    },
    cfg=CONFIG,
    max_metric_rows=12,
)
display_stage_report(stage6_8_report)
display(Image(filename=str(stage6_8_overview_figure)))


<a id="stage-7"></a>

## Stage 7: Tribe Profiling

Turns the frozen Stage 6.8 evidence into stakeholder-ready tribe profiles. No assignments change here — this is pure interpretation.

The 15 **promoted tribes** (✓ strong, ~ usable) are the final deliverable. The 7 **review tribes** are included as hypotheses and clearly labelled throughout.

### Stage 7.0: Evidence Preflight

Generate the Stage 7 artifact pack from the Stage 6.8 evidence bundle. This confirms the read-only evidence boundary, promoted tribe count, potential review tribe count, remaining-customer coverage, visual evidence paths, and handoff paths before interpretation starts.


In [ ]:
import polars as pl
from pathlib import Path
from IPython.display import Image, Markdown, display

from src.profiling import (
    profile_quality_summary,
    write_stage7_final_handoff_pack,
)
from src.stage_reports import display_stage_report, write_stage_report
from src.visualization import (
    plot_stage7_customer_coverage_bar,
    plot_stage7_remaining_customer_segments,
    plot_stage7_relationship_heatmap,
    plot_stage7_review_blockers,
    plot_stage7_soft_audience_opportunities,
    plot_tribe_profile_comparison_heatmap,
    plot_tribe_theme_lift_heatmap,
    plot_tribe_vs_population_evidence_dashboard,
)

stage6_ranked = pl.read_parquet(stage6_diagnostics["parquet"]).sort("stage6_rank")
stage6_winner = stage6_ranked.row(0, named=True)
selected_key = stage6_winner["candidate_id"]

stage7_artifacts_dir = CONFIG.artifacts / "stage7"
stage7_figures_dir = CONFIG.figures
stage7_artifacts_dir.mkdir(parents=True, exist_ok=True)
stage7_figures_dir.mkdir(parents=True, exist_ok=True)

selected_profile_path = stage6_8_tribe_evidence_path
stage68_manifest_path = stage6_8_manifest_path
stage7_quality = profile_quality_summary(selected_profile_path, cfg=CONFIG)
stage7_activation_assignment_path = stage6_activation_assignment_path if "stage6_activation_assignment_path" in globals() else stage6_assignment_path
stage7_behavior_path = behavior_path if "behavior_path" in globals() else None

stage7_final_pack = write_stage7_final_handoff_pack(
    selected_profile_path,
    assignments_path=stage7_activation_assignment_path,
    behavior_path=stage7_behavior_path,
    stage68_manifest_path=stage68_manifest_path,
    output_dir=stage7_artifacts_dir / "final_handoff",
    write_cards=True,
    cfg=CONFIG,
)
stage7_final_index = stage7_final_pack["final_index"]
stage7_card_paths = stage7_final_pack["card_paths"]
stage7_card_count = len(stage7_card_paths)
stage7_final_tribes = stage7_final_index.height
stage7_review_candidates = stage7_final_pack["review_candidates"]
stage7_review_clusters = stage7_review_candidates.height

stage7_substage_paths = stage7_final_pack["substage_paths"]
stage7_all_tribe_profiles = pl.read_csv(stage7_substage_paths["all_tribe_profiles"]["csv"])
stage7_promoted_tribe_validation = pl.read_csv(stage7_substage_paths["promoted_tribe_validation"]["csv"])
stage7_review_tribe_validation = pl.read_csv(stage7_substage_paths["review_tribe_validation"]["csv"])
stage7_all_tribe_product_identity = pl.read_csv(stage7_substage_paths["all_tribe_product_identity"]["csv"])
stage7_all_tribe_behavior_differentiation = pl.read_csv(stage7_substage_paths["all_tribe_behavior_differentiation"]["csv"])
stage7_review_tribe_audit = pl.read_csv(stage7_substage_paths["review_tribe_audit"]["csv"])
stage7_remaining_customer_analysis = pl.read_csv(stage7_substage_paths["remaining_customer_analysis"]["csv"])
stage7_soft_audience_opportunities = pl.read_csv(stage7_substage_paths["soft_audience_opportunities"]["csv"])
stage7_relationship_atlas = pl.read_csv(stage7_substage_paths["all_tribe_relationship_atlas"]["csv"])
stage7_all_tribe_dossiers_path = Path(stage7_substage_paths["all_tribe_dossiers"]["markdown"])
stage7_persona_deep_dives_path = stage7_all_tribe_dossiers_path
stage7_stakeholder_readiness_paths = stage7_final_pack["stakeholder_readiness_paths"]
stage7_campaign_playbook_paths = stage7_final_pack["campaign_playbook_paths"]
stage7_stakeholder_readiness = pl.read_csv(stage7_stakeholder_readiness_paths["csv"])
stage7_campaign_playbook = pl.read_csv(stage7_campaign_playbook_paths["csv"])
stage7_soft_activation_customers_path = stage7_final_pack["soft_audience_activation_customers_csv"]
stage7_soft_activation_customers = pl.read_csv(stage7_soft_activation_customers_path)
stage7_customer_metric_tests_path = stage7_final_pack["customer_metric_tests_csv"]
stage7_customer_metric_tests = pl.read_csv(stage7_customer_metric_tests_path)
stage7_llm_evidence_path = stage7_final_pack["llm_evidence_csv"]
q_threshold = float(CONFIG.get("profiling.significance_q_threshold", 0.05))
stage7_significant_customer_metric_tests = (
    stage7_customer_metric_tests.filter(pl.col("anova_q_value") <= q_threshold).height
    if not stage7_customer_metric_tests.is_empty() and "anova_q_value" in stage7_customer_metric_tests.columns
    else 0
)

tribe_vs_population_dashboard = plot_tribe_vs_population_evidence_dashboard(
    selected_profile_path,
    output_path=stage7_figures_dir / f"stage7_tribe_vs_population_evidence_dashboard_{CONFIG.mode}.png",
    assignment_path=stage7_activation_assignment_path,
    cfg=CONFIG,
)
stage7_product_theme_heatmap = plot_tribe_theme_lift_heatmap(
    selected_profile_path,
    output_path=stage7_figures_dir / f"stage7_all_tribe_product_theme_lift_heatmap_{CONFIG.mode}.png",
    cfg=CONFIG,
)
stage7_behavior_heatmap = plot_tribe_profile_comparison_heatmap(
    selected_profile_path,
    output_path=stage7_figures_dir / f"stage7_all_tribe_behavior_metric_heatmap_{CONFIG.mode}.png",
    cfg=CONFIG,
)
stage7_relationship_heatmap = plot_stage7_relationship_heatmap(
    stage7_relationship_atlas,
    output_path=stage7_figures_dir / f"stage7_all_tribe_relationship_heatmap_{CONFIG.mode}.png",
    cfg=CONFIG,
)

stage7_model_table = pl.DataFrame([
    {
        "stage6_rank": stage6_winner.get("stage6_rank"),
        "candidate_id": selected_key,
        "cluster_count": stage6_winner.get("cluster_count"),
        "profile_path": str(selected_profile_path),
        "final_story": str(stage7_final_pack["story_markdown"]),
        "final_index": str(stage7_final_pack["index_csv"]),
        "tribe_cards": str(stage7_final_pack["card_directory"]),
    }
])
stage7_scope_table = pl.DataFrame([
    {
        "total_profiled_tribes": int(stage7_all_tribe_profiles.height),
        "promoted_tribes": int(stage7_final_tribes),
        "review_tribes": int(stage7_review_clusters),
        "remaining_customer_segments": int(stage7_remaining_customer_analysis.height),
        "soft_audience_opportunities": int(stage7_soft_audience_opportunities.height),
        "relationship_pairs": int(stage7_relationship_atlas.height),
        "stakeholder_readiness_status": stage7_stakeholder_readiness.filter(pl.col("check_id") == "overall_delivery_readiness")[0, "status"] if not stage7_stakeholder_readiness.is_empty() else "missing",
        "stakeholder_readiness_critical_findings": stage7_stakeholder_readiness.filter((pl.col("status") == "fail") & (pl.col("severity") == "critical")).height if not stage7_stakeholder_readiness.is_empty() else 0,
        "soft_activation_customer_rows": int(stage7_soft_activation_customers.height),
    }
])
stage7_report = write_stage_report(
    "07",
    "Tribe Profiling and Communication",
    summary=[
        f"Profiling selected Stage 6 model: {selected_key}",
        "Accepted Stage 6.6 readiness as final; final promotion is separated in Stage 7.2, while Stage 7.3 onward profiles every retained tribe with status labels.",
        "Read Stage 6.8 core-only product evidence plus per-tribe transaction/customer exports; use the rescued assignment only for audience sizing and activation coverage.",
        "Only Stage 7.2 separates final promoted tribes from potential review tribes; Stage 7.3 onward includes every retained tribe with a status label.",
        "Review tribes receive full potential-tribe profiles and are explicitly marked potential_review.",
        "Cards and dossier rows combine core-only product evidence, widely purchased products, sector spend share, spend percentiles, recency, temporal, and loyalty context.",
    ],
    metrics=[
        ("profiled_candidate_tribes", stage7_quality.get("profiled_clusters"), "all retained Stage 6 clusters reviewed"),
        ("final_promoted_tribes", stage7_final_tribes, "Stage 6.6 promoted readiness only"),
        ("tribe_cards", stage7_card_count, "one PNG card per retained tribe; review cards are potential/review"),
        ("review_candidate_clusters", stage7_review_clusters, "held out and recorded in manifest"),
        ("all_tribe_profile_rows", stage7_all_tribe_profiles.height, "promoted plus review-only tribes"),
        ("remaining_customer_segments", stage7_remaining_customer_analysis.height, "descriptive groups outside hard tribes"),
        ("soft_audience_opportunities", stage7_soft_audience_opportunities.height, "campaign-use only; no assignment mutation"),
        ("tribe_relationship_pairs", stage7_relationship_atlas.height, "pairwise similarity/difference map"),
        ("clusters_with_significant_product_lift", stage7_quality.get("clusters_with_significant_product_lift"), "context only; not a Stage 7 gate"),
        ("avg_significant_product_lifts_per_cluster", stage7_quality.get("avg_significant_product_lifts_per_cluster"), "context only"),
        ("avg_max_product_lift_vs_rest", stage7_quality.get("avg_max_product_lift_vs_rest"), ">1.5 indicates clear over-indexing"),
        ("customer_metric_anova_tests", stage7_customer_metric_tests.height, "precomputed in Stage 6.8"),
        ("significant_customer_metric_anova_tests", stage7_significant_customer_metric_tests, "q<=0.05 after FDR in Stage 6.8"),
        ("stage68_product_lift_rows", stage6_8_counts.get("product_lifts"), "precomputed in Stage 6.8"),
        ("stage68_transaction_export_files", len(stage6_8_manifest.get("transaction_exports", [])), "precomputed in Stage 6.8"),
        ("stakeholder_readiness_critical_findings", stage7_stakeholder_readiness.filter((pl.col("status") == "fail") & (pl.col("severity") == "critical")).height if not stage7_stakeholder_readiness.is_empty() else 0, "Stage 7.9 delivery gate"),
        ("soft_activation_customer_rows", stage7_soft_activation_customers.height, "customer-level campaign-use rows"),
    ],
    figures={
        "Tribe vs population dashboard": tribe_vs_population_dashboard,
        "All-tribe product-theme heatmap": stage7_product_theme_heatmap,
        "All-tribe behavior heatmap": stage7_behavior_heatmap,
        "All-tribe relationship heatmap": stage7_relationship_heatmap,
        "Tribe card directory": stage7_final_pack["card_directory"],
    },
    artifacts={
        "Final story MD": stage7_final_pack["story_markdown"],
        "Final story HTML": stage7_final_pack["story_html"],
        "Final index CSV": stage7_final_pack["index_csv"],
        "Final manifest": stage7_final_pack["manifest_json"],
        "Selected core profile parquet": selected_profile_path,
        "Stage 6.8 manifest": stage68_manifest_path,
        "Stage 6 core-only profiling assignment": stage6_profile_assignment_path if "stage6_profile_assignment_path" in globals() else selected_profile_path,
        "Stage 6 activation/rescued assignment": stage7_activation_assignment_path,
        "All-tribe product summary long table": stage7_final_pack["product_summary_paths"]["combined_csv"],
        "Evidence long CSV": stage7_llm_evidence_path,
        "All-tribe comparison HTML": stage7_final_pack["comparison_paths"]["html"],
        "Customer metric context": stage7_customer_metric_tests_path,
        "All-tribe profiles": stage7_substage_paths["all_tribe_profiles"]["csv"],
        "Promoted tribe validation": stage7_substage_paths["promoted_tribe_validation"]["csv"],
        "Review tribe validation": stage7_substage_paths["review_tribe_validation"]["csv"],
        "All-tribe product identity": stage7_substage_paths["all_tribe_product_identity"]["csv"],
        "All-tribe behavior differentiation": stage7_substage_paths["all_tribe_behavior_differentiation"]["csv"],
        "All-tribe dossiers": stage7_substage_paths["all_tribe_dossiers"]["markdown"],
        "Remaining customer analysis": stage7_substage_paths["remaining_customer_analysis"]["csv"],
        "Soft audience opportunities": stage7_substage_paths["soft_audience_opportunities"]["csv"],
        "Soft audience activation customers": stage7_soft_activation_customers_path,
        "Campaign playbook": stage7_campaign_playbook_paths["csv"],
        "Stakeholder readiness": stage7_stakeholder_readiness_paths["csv"],
        "All-tribe relationship atlas": stage7_substage_paths["all_tribe_relationship_atlas"]["csv"],
    },
    cfg=CONFIG,
    max_metric_rows=14,
)

display_stage_report(stage7_report, max_lines=80)
display(stage7_scope_table)
display(stage7_model_table)
display(stage7_final_index)


### Stage 7.1: Customer Coverage

Where do all 1.48M customers go?

- **Promoted tribes** (15): hard HDBSCAN core members assigned to a validated tribe
- **Review tribes** (7): hard members in lower-confidence clusters, kept as hypotheses
- **Rescued / soft-assigned**: centroid-proximity rescue — counted in coverage, excluded from profiling
- **Remaining noise**: genuinely unassigned customers (broad generalists or weak-signal shoppers)

In [ ]:
_centroid_rescued = int((stage6_noise_rescue_stats or {}).get("rescued_customers", 0)) if "stage6_noise_rescue_stats" in globals() else 0
_still_noise = int((stage6_noise_rescue_stats or {}).get("still_noise_customers", 0)) if "stage6_noise_rescue_stats" in globals() else 0

stage7_coverage_table = pl.DataFrame([
    {
        "coverage_group": "final_promoted_tribes",
        "tribes": stage7_final_tribes,
        "customers": int(stage7_final_index["customers"].sum()) if not stage7_final_index.is_empty() else 0,
        "membership_policy": "hard HDBSCAN core — final stakeholder tribes",
    },
    {
        "coverage_group": "potential_review_tribes",
        "tribes": stage7_review_clusters,
        "customers": int(stage7_review_candidates["customers"].sum()) if not stage7_review_candidates.is_empty() and "customers" in stage7_review_candidates.columns else 0,
        "membership_policy": "hard HDBSCAN core — potential tribes; not final validation",
    },
    {
        "coverage_group": "centroid_rescued",
        "tribes": None,
        "customers": _centroid_rescued,
        "membership_policy": "centroid-proximity rescue — counted in coverage, excluded from profiling",
    },
    {
        "coverage_group": "remaining_noise_segments",
        "tribes": None,
        "customers": _still_noise if _still_noise else int(stage7_remaining_customer_analysis["customer_count"].sum()) if not stage7_remaining_customer_analysis.is_empty() else 0,
        "membership_policy": "genuinely unassigned after rescue — descriptive segments only",
    },
])
stage7_coverage_figure = plot_stage7_customer_coverage_bar(
    stage7_coverage_table,
    output_path=stage7_figures_dir / f"stage7_customer_coverage_{CONFIG.mode}.png",
    cfg=CONFIG,
)
display(stage7_coverage_table)
display(Image(filename=str(stage7_coverage_figure)))
display(stage7_scope_table)


### Stage 7.2A: Promoted Tribe Validation

The 15 promoted tribes and their quality evidence. ✓ Strong tribes are fully validated; ~ Usable tribes are promoted with a stability caveat.

The **per-tribe evidence panel** (figure below) shows each tribe's size, top product and sector lift, evidence count, and behavioral fingerprint versus the rest of the assigned population.

In [ ]:
display(stage7_promoted_tribe_validation)
display(Image(filename=str(tribe_vs_population_dashboard)))


### Stage 7.2B: Review Tribe Audit

The 7 review tribes failed at least one quality gate — typically too few core customers, low jitter recovery, or low assignment confidence. They are **not final tribes**.

**Why they are still shown:** several have strong product-lift evidence and real purchasing signal. They are included as hypotheses that a future run (more data, relaxed size threshold) could validate.

The **blockers chart** shows which specific quality gate each review tribe failed.

In [ ]:
from src.visualization import plot_stage7_review_tribe_jitter

display(stage7_review_tribe_validation)
# Jitter label-recovery accuracy per review tribe — shows why each is held, not promoted
stage7_review_jitter_figure = plot_stage7_review_tribe_jitter(
    stage7_review_tribe_validation,
    stage6_6_cluster_readiness,
    output_path=stage7_figures_dir / f"stage7_review_tribe_jitter_{CONFIG.mode}.png",
    cfg=CONFIG,
)
display(Image(filename=str(stage7_review_jitter_figure)))
if not stage7_review_candidates.is_empty():
    display(stage7_review_candidates)


### Stage 7.3: What Each Tribe Buys

The table shows each tribe's top products and sectors. The **heatmap** below maps every tribe against the top product themes, ranked by strongest over-index.

> **How to read the heatmap:** Each cell value is the lift — `2.5x` means this tribe buys that product theme 2.5x more than the average customer. Darker blue = stronger over-index. Blank cells = near average or below the significance threshold.

In [ ]:
display(stage7_all_tribe_product_identity)
display(Image(filename=str(stage7_product_theme_heatmap)))


### Stage 7.4: How Each Tribe Shops

The table compares spending, visit frequency, basket size, and promotion use across tribes. The **heatmap** below shows the same data visually.

> **How to read the heatmap:** Each cell = tribe rate divided by all other shoppers' rate. `2.0x` = tribe does this twice as much as others. `0.5x` = half the average. `--` = near average. Warm green = tribe over-indexes here. Blue/purple = under-indexes.

In [ ]:
display(stage7_all_tribe_behavior_differentiation)
display(Image(filename=str(stage7_behavior_heatmap)))


### Stage 7.5: All-Tribe Dossiers

One concise dossier per retained tribe: top products, behavioral summary, a plain-language description, and a caveat where applicable.

Final promoted tribes are ready for stakeholder use. Review tribes are labelled **Potential / Review** and should be treated as hypotheses only.

In [ ]:
display(Markdown(stage7_all_tribe_dossiers_path.read_text(encoding="utf-8")))
first_card = next(iter(stage7_card_paths.values()), None)
if first_card is not None:
    display(Image(filename=str(first_card)))


### Stage 7.6: Remaining Customer Analysis

Customers still outside hard tribes after rescue. Segmented into descriptive groups (broad generalists, weak-signal, near-tribe fringe, etc.) for context — these are not tribe memberships.

In [ ]:
stage7_remaining_customer_figure = plot_stage7_remaining_customer_segments(
    stage7_remaining_customer_analysis,
    output_path=stage7_figures_dir / f"stage7_remaining_customer_segments_{CONFIG.mode}.png",
    cfg=CONFIG,
)
display(stage7_remaining_customer_analysis)
display(Image(filename=str(stage7_remaining_customer_figure)))


### Stage 7.7: Campaign-Use Soft Audiences

**Where this comes from:** centroid-rescued customers (Stage 6.5a) who landed near a specific tribe. Their top affinity score and margin to the next-closest tribe are used to filter for high-confidence assignments.

**What it shows:** for each tribe, how many rescued customers can be targeted in a campaign as a soft audience. These are not tribe members — they are activation-ready lookalikes for reach campaigns.

> Use these audiences for CRM / activation tests, not for segmentation reporting.

In [ ]:
stage7_activation_summary = (
    stage7_soft_activation_customers.group_by(["target_tribe_id", "target_tribe_name", "target_tribe_status"])
    .agg(
        pl.len().alias("activation_customers"),
        pl.col("top_affinity_score").mean().alias("mean_top_affinity"),
        pl.col("affinity_margin").mean().alias("mean_affinity_margin"),
    )
    .sort("activation_customers", descending=True)
    if not stage7_soft_activation_customers.is_empty()
    else pl.DataFrame(
        schema={
            "target_tribe_id": pl.Int64,
            "target_tribe_name": pl.Utf8,
            "target_tribe_status": pl.Utf8,
            "activation_customers": pl.Int64,
            "mean_top_affinity": pl.Float64,
            "mean_affinity_margin": pl.Float64,
        }
    )
)
if stage7_soft_audience_opportunities.is_empty():
    display(Markdown("No high-confidence campaign-use soft audience opportunities were produced."))
else:
    display(stage7_soft_audience_opportunities)
stage7_soft_audience_figure = plot_stage7_soft_audience_opportunities(
    stage7_soft_audience_opportunities,
    output_path=stage7_figures_dir / f"stage7_soft_audience_opportunities_{CONFIG.mode}.png",
    cfg=CONFIG,
)
display(Image(filename=str(stage7_soft_audience_figure)))
display(stage7_activation_summary)


### Stage 7.8: Tribe Relationships

Shows which tribes share purchasing patterns — useful for cross-sell strategy or understanding where two tribes overlap.

> **How to read the heatmap:** Each cell = combined product-overlap and behavioral-similarity score. A score near `0.6` means strong shared behavior; near `0` means fully distinct tribes. The diagonal (a tribe with itself) is masked. Only scores ≥ 0.35 are labelled.

In [ ]:
display(stage7_relationship_atlas.head(30))
display(Image(filename=str(stage7_relationship_heatmap)))


### Stage 7.9: Stakeholder Synthesis And Delivery Readiness

Collect the final story and delivery artifacts. Final promoted tribes, potential review tribes, and remaining/noise customers are summarized separately before the delivery readiness gate runs.


In [ ]:
stage7_critical_findings = stage7_stakeholder_readiness.filter(
    (pl.col("status") == "fail") & (pl.col("severity") == "critical")
)
stage7_warning_findings = stage7_stakeholder_readiness.filter(pl.col("status") == "warn")

display(Markdown(
    f"### Stage 7 Stakeholder Handoff\n\n"
    f"Final story: {stage7_final_pack['story_markdown']}  \\n"
    f"Readable HTML: {stage7_final_pack['story_html']}  \\n"
    f"Tribe cards: {stage7_final_pack['card_directory']}  \\n"
    f"Final promoted tribes: {stage7_final_tribes}; potential review tribes: {stage7_review_clusters}.  \\n"
    f"Evidence bundle: {stage68_manifest_path}  \\n"
    "Use final promoted tribes for stakeholder segmentation; use potential review tribes only as hypothesis/test audiences."
))
display(Markdown(
    "### Supporting Evidence\n\n"
    f"- Final index: {stage7_final_pack['index_csv']}\n"
    f"- All-tribe product identity: {stage7_substage_paths['all_tribe_product_identity']['csv']}\n"
    f"- All-tribe behavior differentiation: {stage7_substage_paths['all_tribe_behavior_differentiation']['csv']}\n"
    f"- All-tribe dossiers: {stage7_substage_paths['all_tribe_dossiers']['markdown']}\n"
    f"- All-tribe relationship atlas: {stage7_substage_paths['all_tribe_relationship_atlas']['csv']}\n"
    f"- Remaining customer analysis: {stage7_substage_paths['remaining_customer_analysis']['csv']}\n"
    f"- Manifest: {stage7_final_pack['manifest_json']}"
))
display(stage7_stakeholder_readiness)
if not stage7_warning_findings.is_empty():
    display(Markdown("### Stage 7.9 Warnings"))
    display(stage7_warning_findings)
display(Markdown("### Stage 7.9 Final-Only Campaign Playbook"))
display(stage7_campaign_playbook)
display(Markdown(
    f"Readiness report: {stage7_stakeholder_readiness_paths['csv']}  \\n"
    f"Campaign playbook: {stage7_campaign_playbook_paths['csv']}  \\n"
    f"Activation customers CSV: {stage7_soft_activation_customers_path}  \\n"
    f"Activation customers parquet: {stage7_final_pack['soft_audience_activation_customers_parquet']}"
))

fail_on_critical = bool(CONFIG.get("profiling.stage7_delivery_readiness.fail_on_critical", True))
if fail_on_critical and not stage7_critical_findings.is_empty():
    raise ValueError(
        "Stage 7.9 stakeholder readiness failed. "
        f"Critical findings: {stage7_critical_findings['check_id'].to_list()}"
    )


<a id="stage-8"></a>

## Stage 8: Dashboard Data Pack

Builds and validates all dashboard-ready tables. Counts, schemas, and headline numbers are checked before anything reaches the presentation layer.

In [ ]:
import importlib
import json
from pathlib import Path

import polars as pl
from IPython.display import Markdown, display

import src.stage8
importlib.reload(src.stage8)
from src.stage8 import STAGE8_SCHEMA_VERSION, stage8_artifact_paths, write_stage8_dashboard_pack

# Equivalent terminal command: CARREFOUR_MODE=prod python -m src.stage8
stage8_outputs = write_stage8_dashboard_pack(cfg=CONFIG)
stage8_paths = stage8_artifact_paths(cfg=CONFIG)
stage8_manifest_path = stage8_paths["dashboard_manifest"]
stage8_manifest = json.loads(stage8_manifest_path.read_text(encoding="utf-8"))
stage8_products = stage8_manifest["created_products"]

stage8_inventory_rows = []
for product in stage8_products:
    path = Path(product["path"])
    schema = product.get("schema") or []
    pages = product.get("dashboard_pages") or []
    stage8_inventory_rows.append(
        {
            "name": product["name"],
            "format": product.get("format"),
            "exists": path.exists(),
            "row_count": product.get("row_count"),
            "column_count": len(schema),
            "dashboard_pages": "; ".join(pages),
            "schema_preview": ", ".join(schema[:12]) + (" ..." if len(schema) > 12 else ""),
            "size_mb": round(path.stat().st_size / 1_000_000, 3) if path.exists() else None,
            "path": str(path),
        }
    )

stage8_artifact_inventory = pl.DataFrame(stage8_inventory_rows, infer_schema_length=None).sort(["format", "name"])
stage8_table_artifacts = stage8_artifact_inventory.filter(pl.col("format").is_in(["parquet", "csv"]))
stage8_non_table_artifacts = stage8_artifact_inventory.filter(~pl.col("format").is_in(["parquet", "csv"]))

stage8_readiness = pl.read_csv(stage8_paths["readiness_csv"])
stage8_readiness_status = stage8_manifest.get("readiness_summary", {})

display(Markdown(
    f"### Stage 8 Build Summary\n\n"
    f"Manifest: `{stage8_manifest_path}`  \n"
    f"Schema version: `{stage8_manifest.get('schema_version')}` / expected `{STAGE8_SCHEMA_VERSION}`  \n"
    f"Artifacts created: `{len(stage8_products)}`  \n"
    f"Table artifacts: `{stage8_table_artifacts.height}`  \n"
    f"Readiness: `{stage8_readiness_status}`"
))
display(stage8_artifact_inventory.select([
    "name", "format", "exists", "row_count", "column_count", "size_mb", "dashboard_pages", "schema_preview"
]))
display(Markdown("### Stage 8 Readiness Report"))
display(stage8_readiness)


In [ ]:
EXPECTED_STAGE8_COUNTS = {
    "total_customers": 1_478_831,
    "retained_hard_tribes": 22,
    "final_promoted_tribes": 15,
    "review_tribes": 7,
    "final_customers": 723_207,  # includes centroid-rescued (hard core + soft assigned)
    "review_customers": 647_990,  # includes centroid-rescued in review tribes
    "remaining_customers": 107_634,  # truly unassigned after centroid rescue
    "remaining_segments": 6,
}

stage8_dim_tribe = pl.read_parquet(stage8_paths["rel_dim_tribe"])
stage8_fact_tribe_metrics = pl.read_parquet(stage8_paths["rel_fact_tribe_metrics"])
stage8_fact_tribe_product = pl.read_parquet(stage8_paths["rel_fact_tribe_product_affinity"])
stage8_dim_action_target = pl.read_parquet(stage8_paths["rel_dim_action_target"])
stage8_fact_segment_action = pl.read_parquet(stage8_paths["rel_fact_segment_action"])
stage8_fact_coverage_metrics = pl.read_parquet(stage8_paths["rel_fact_coverage_group_metrics"])
stage8_dim_coverage_group = pl.read_parquet(stage8_paths["rel_dim_coverage_group"])
stage8_fact_remaining_segment_metrics = pl.read_parquet(stage8_paths["rel_fact_remaining_segment_metrics"])
stage8_embedding_sample_assignments = (
    pl.read_parquet(stage8_paths["rel_fact_embedding_3d_sample"])
    .join(
        pl.read_parquet(stage8_paths["rel_fact_customer_assignment"], columns=["cliente", "tribe_id", "coverage_group"]),
        on="cliente",
        how="left",
    )
)

stage8_total_customers = int(stage8_fact_coverage_metrics.select(pl.col("customers").sum()).item())
stage8_final_customers = int(stage8_fact_coverage_metrics.filter(pl.col("coverage_group") == "core_promoted_tribe").select(pl.col("customers").sum()).item())
stage8_review_customers = int(stage8_fact_coverage_metrics.filter(pl.col("coverage_group") == "review_tribe").select(pl.col("customers").sum()).item())
stage8_remaining_customers = stage8_total_customers - stage8_final_customers - stage8_review_customers
stage8_retained_hard_customers = int(stage8_fact_tribe_metrics.select(pl.col("assigned_customers").sum()).item())  # hard + centroid-rescued
stage8_final_tribes = int(stage8_dim_tribe.filter(pl.col("is_final_tribe")).height)
stage8_review_tribes = int(stage8_dim_tribe.filter(pl.col("is_review_tribe")).height)

stage8_assignment_uniqueness = pl.scan_parquet(stage8_paths["rel_fact_customer_assignment"]).select(
    pl.len().alias("rows"),
    pl.col("cliente").n_unique().alias("unique_clientes"),
).collect().row(0, named=True)
stage8_value_behavior_uniqueness = pl.scan_parquet(stage8_paths["rel_fact_customer_value_behavior"]).select(
    pl.len().alias("rows"),
    pl.col("cliente").n_unique().alias("unique_clientes"),
).collect().row(0, named=True)
stage8_embedding_uniqueness = pl.scan_parquet(stage8_paths["rel_fact_embedding_3d"]).select(
    pl.len().alias("rows"),
    pl.col("cliente").n_unique().alias("unique_clientes"),
    pl.all_horizontal([pl.col(c).is_finite().fill_null(False) for c in ["x", "y", "z"]]).all().alias("xyz_finite"),
).collect().row(0, named=True)
stage8_embedding_sample_check = pl.scan_parquet(stage8_paths["rel_fact_embedding_3d_sample"]).select(
    pl.len().alias("rows"),
    pl.col("cliente").n_unique().alias("unique_clientes"),
    pl.all_horizontal([pl.col(c).is_finite().fill_null(False) for c in ["x", "y", "z"]]).all().alias("xyz_finite"),
).collect().row(0, named=True)

stage8_manifest_missing_outputs = sorted(
    product["name"] for product in stage8_products if not Path(product["path"]).exists()
)
stage8_markdown_inputs = sorted(
    key for key, value in stage8_manifest.get("input_paths", {}).items()
    if Path(value).suffix.lower() in {".md", ".markdown"}
)
stage8_manifest_table_names = set(stage8_table_artifacts["name"].to_list())
stage8_required_table_names = {
    "artifact_manifest", "relational_integrity_report",
    "rel_dim_tribe", "rel_dim_tribe_name_proposal", "rel_dim_coverage_group", "rel_dim_remaining_segment",
    "rel_dim_product", "rel_dim_category", "rel_dim_action_target", "rel_dim_dashboard_page", "rel_dim_visualization",
    "rel_dim_pipeline_stage", "rel_dim_model_card", "rel_dim_feature_group", "rel_dim_story_chapter", "rel_dim_executive_insight",
    "rel_fact_executive_metric", "rel_fact_customer_assignment", "rel_fact_customer_value_behavior", "rel_fact_customer_nearest_tribe",
    "rel_fact_tribe_metrics", "rel_fact_tribe_profile_text", "rel_fact_coverage_group_metrics", "rel_fact_remaining_segment_metrics",
    "rel_fact_tribe_product_affinity", "rel_fact_tribe_category_affinity", "rel_fact_segment_action",
    "rel_bridge_tribe_similarity", "rel_fact_embedding_2d", "rel_fact_embedding_3d", "rel_fact_embedding_3d_sample",
    "rel_fact_embedding_3d_pca", "rel_fact_embedding_3d_pca_sample", "rel_fact_embedding_centroid",
    "rel_fact_validation_metric", "rel_fact_data_quality_metric",
}
stage8_missing_required_tables = sorted(stage8_required_table_names - stage8_manifest_table_names)
stage8_forbidden_legacy_products = {
    "tribe_master", "tribe_profiles", "tribe_deep_dive", "tribe_products", "tribe_categories",
    "customer_assignments", "customer_coverage", "customer_coverage_summary", "remaining_customer_segments",
    "segment_actions", "embedding_2d", "embedding_3d", "embedding_3d_sample", "model_cards",
    "pipeline_stages", "feature_catalog", "validation_metrics", "data_quality_metrics", "dashboard_page_contracts",
    "visualization_specs", "missing_file_register",
}
stage8_present_legacy_products = sorted(stage8_forbidden_legacy_products & set(item["name"] for item in stage8_products))

stage8_master_tribes = set(stage8_dim_tribe["tribe_id"].to_list())
stage8_product_tribes = set(stage8_fact_tribe_product.select("tribe_id").unique()["tribe_id"].to_list())
stage8_missing_product_evidence = sorted(stage8_master_tribes - stage8_product_tribes)
stage8_promoted_tribes = set(stage8_dim_tribe.filter(pl.col("is_final_tribe"))["tribe_id"].to_list())
stage8_action_fact_targets = set(stage8_fact_segment_action["target_id"].to_list())
stage8_action_tribes = set(
    stage8_dim_action_target
    .filter((pl.col("target_type") == "tribe") & pl.col("tribe_id").is_not_null() & pl.col("target_id").is_in(stage8_action_fact_targets))
    .select("tribe_id")
    .unique()["tribe_id"]
    .to_list()
)
stage8_missing_promoted_actions = sorted(stage8_promoted_tribes - stage8_action_tribes)
stage8_sample_tribes = set(stage8_embedding_sample_assignments.filter(pl.col("tribe_id").is_not_null())["tribe_id"].to_list())
stage8_sample_groups = set(stage8_embedding_sample_assignments["coverage_group"].to_list())
stage8_coverage_groups = set(stage8_fact_coverage_metrics["coverage_group"].to_list())
stage8_missing_sample_tribes = sorted(stage8_master_tribes - stage8_sample_tribes)
stage8_missing_sample_groups = sorted(stage8_coverage_groups - stage8_sample_groups)
stage8_readiness_critical_failures = stage8_readiness.filter(
    (pl.col("status") == "fail") & (pl.col("severity") == "critical")
).height
stage8_relational_integrity = pl.read_parquet(stage8_paths["relational_integrity_report"])
stage8_relational_integrity_failures = stage8_relational_integrity.filter(pl.col("status") == "fail").height


def stage8_check(check_id, area, severity, passed, details):
    return {
        "check_id": check_id,
        "area": area,
        "severity": severity,
        "status": "pass" if passed else "fail",
        "details": details,
    }

stage8_sanity_rows = [
    stage8_check("manifest_paths_exist", "manifest", "critical", not stage8_manifest_missing_outputs, f"missing={stage8_manifest_missing_outputs}"),
    stage8_check("schema_version", "manifest", "critical", stage8_manifest.get("schema_version") == STAGE8_SCHEMA_VERSION, f"actual={stage8_manifest.get('schema_version')}, expected={STAGE8_SCHEMA_VERSION}"),
    stage8_check("no_markdown_input_dependencies", "contract", "critical", not stage8_markdown_inputs, f"markdown_inputs={stage8_markdown_inputs}"),
    stage8_check("required_relational_tables_present", "contract", "critical", not stage8_missing_required_tables, f"missing={stage8_missing_required_tables}"),
    stage8_check("legacy_wide_products_not_published", "contract", "critical", not stage8_present_legacy_products, f"legacy_present={stage8_present_legacy_products}"),
    stage8_check("readiness_report_has_no_critical_failures", "readiness", "critical", stage8_readiness_critical_failures == 0, f"critical_failures={stage8_readiness_critical_failures}"),
    stage8_check("relational_integrity_has_no_failures", "relational_model", "critical", stage8_relational_integrity_failures == 0, f"failures={stage8_relational_integrity_failures}"),
]

if CONFIG.mode == "prod":
    stage8_sanity_rows.extend([
        stage8_check("total_customer_reconciliation", "coverage", "critical", stage8_total_customers == EXPECTED_STAGE8_COUNTS["total_customers"], f"actual={stage8_total_customers:,}, expected={EXPECTED_STAGE8_COUNTS['total_customers']:,}"),
        stage8_check("final_customer_reconciliation", "coverage", "critical", stage8_final_customers == EXPECTED_STAGE8_COUNTS["final_customers"], f"actual={stage8_final_customers:,}, expected={EXPECTED_STAGE8_COUNTS['final_customers']:,}"),
        stage8_check("review_customer_reconciliation", "coverage", "critical", stage8_review_customers == EXPECTED_STAGE8_COUNTS["review_customers"], f"actual={stage8_review_customers:,}, expected={EXPECTED_STAGE8_COUNTS['review_customers']:,}"),
        stage8_check("remaining_customer_reconciliation", "coverage", "critical", stage8_remaining_customers == EXPECTED_STAGE8_COUNTS["remaining_customers"], f"actual={stage8_remaining_customers:,}, expected={EXPECTED_STAGE8_COUNTS['remaining_customers']:,}"),
        stage8_check("retained_hard_tribe_count", "tribes", "critical", stage8_dim_tribe.height == EXPECTED_STAGE8_COUNTS["retained_hard_tribes"], f"actual={stage8_dim_tribe.height}, expected={EXPECTED_STAGE8_COUNTS['retained_hard_tribes']}"),
        stage8_check("promoted_tribe_count", "tribes", "critical", stage8_final_tribes == EXPECTED_STAGE8_COUNTS["final_promoted_tribes"], f"actual={stage8_final_tribes}, expected={EXPECTED_STAGE8_COUNTS['final_promoted_tribes']}"),
        stage8_check("review_tribe_count", "tribes", "critical", stage8_review_tribes == EXPECTED_STAGE8_COUNTS["review_tribes"], f"actual={stage8_review_tribes}, expected={EXPECTED_STAGE8_COUNTS['review_tribes']}"),
        stage8_check("remaining_segment_count", "remaining_customers", "critical", stage8_fact_remaining_segment_metrics.height == EXPECTED_STAGE8_COUNTS["remaining_segments"], f"actual={stage8_fact_remaining_segment_metrics.height}, expected={EXPECTED_STAGE8_COUNTS['remaining_segments']}"),
    ])
else:
    stage8_sanity_rows.append(stage8_check("non_prod_customer_reconciliation", "coverage", "warn", stage8_total_customers > 0, f"mode={CONFIG.mode}; exact prod counts skipped; total={stage8_total_customers:,}"))

stage8_sanity_rows.extend([
    stage8_check("hard_tribe_customers_equal_final_plus_review", "coverage", "critical", stage8_retained_hard_customers == stage8_final_customers + stage8_review_customers, f"hard={stage8_retained_hard_customers:,}; final+review={stage8_final_customers + stage8_review_customers:,}"),
    stage8_check("customer_assignments_unique_by_cliente", "lookup", "critical", stage8_assignment_uniqueness["rows"] == stage8_assignment_uniqueness["unique_clientes"], str(stage8_assignment_uniqueness)),
    stage8_check("customer_value_behavior_unique_by_cliente", "lookup", "critical", stage8_value_behavior_uniqueness["rows"] == stage8_value_behavior_uniqueness["unique_clientes"], str(stage8_value_behavior_uniqueness)),
    stage8_check("embedding_3d_unique_and_finite", "embedding", "critical", stage8_embedding_uniqueness["rows"] == stage8_embedding_uniqueness["unique_clientes"] and bool(stage8_embedding_uniqueness["xyz_finite"]), str(stage8_embedding_uniqueness)),
    stage8_check("embedding_3d_sample_unique_and_finite", "embedding", "critical", stage8_embedding_sample_check["rows"] == stage8_embedding_sample_check["unique_clientes"] and bool(stage8_embedding_sample_check["xyz_finite"]), str(stage8_embedding_sample_check)),
    stage8_check("embedding_3d_sample_size", "embedding", "critical", (75_000 <= stage8_embedding_sample_check["rows"] <= 100_000) if CONFIG.mode == "prod" else stage8_embedding_sample_check["rows"] > 0, f"sample_rows={stage8_embedding_sample_check['rows']:,}"),
    stage8_check("embedding_sample_contains_all_retained_tribes", "embedding", "critical", not stage8_missing_sample_tribes, f"missing={stage8_missing_sample_tribes}"),
    stage8_check("embedding_sample_contains_all_coverage_groups", "embedding", "critical", not stage8_missing_sample_groups, f"missing={stage8_missing_sample_groups}"),
    stage8_check("every_retained_tribe_has_product_evidence", "tribes", "critical", not stage8_missing_product_evidence, f"missing={stage8_missing_product_evidence}"),
    stage8_check("every_promoted_tribe_has_action", "activation", "critical", not stage8_missing_promoted_actions, f"missing={stage8_missing_promoted_actions}"),
])

stage8_sanity_checks = pl.DataFrame(stage8_sanity_rows).sort(["severity", "area", "check_id"])
stage8_failed_critical = stage8_sanity_checks.filter((pl.col("status") == "fail") & (pl.col("severity") == "critical"))

stage8_coverage_display = (
    stage8_fact_coverage_metrics
    .join(stage8_dim_coverage_group, on="coverage_group", how="left")
    .sort("sort_order")
)
stage8_tribe_readiness_snapshot = (
    stage8_dim_tribe
    .join(stage8_fact_tribe_metrics, on="tribe_id", how="left")
    .select([
        "tribe_id", "display_name", "display_technical_name", "promotion_decision",
        "assigned_customers", "hard_assigned_customers", "soft_assigned_customers",
        "population_share_pct", "mean_assignment_confidence", "jitter_label_recovery_accuracy",
        "business_confidence", "readiness_caveat"
    ])
    .sort("tribe_id")
)

display(Markdown("### Stage 8 Sanity Checks"))
display(stage8_sanity_checks)

display(Markdown("### Coverage Reconciliation"))
display(stage8_coverage_display.select([
    "coverage_group", "coverage_group_label", "customers", "customer_share_pct", "revenue", "revenue_share_pct", "recommended_treatment"
]))

display(Markdown("### Tribe Relational Readiness Snapshot"))
display(stage8_tribe_readiness_snapshot)

if not stage8_failed_critical.is_empty():
    raise ValueError(
        "Stage 8 relational artifact sanity checks failed: "
        f"{stage8_failed_critical['check_id'].to_list()}"
    )


In [ ]:
STAGE8_TABLE_PREVIEW_ROWS = 3
STAGE8_MAX_STRING_CHARS = 140


def stage8_compact_preview(df: pl.DataFrame, max_chars: int = STAGE8_MAX_STRING_CHARS) -> pl.DataFrame:
    preview_exprs = []
    for column_name, dtype in df.schema.items():
        if dtype == pl.Utf8 or str(dtype) == "String":
            preview_exprs.append(
                pl.when(pl.col(column_name).str.len_chars() > max_chars)
                .then(pl.col(column_name).str.slice(0, max_chars) + "...")
                .otherwise(pl.col(column_name))
                .alias(column_name)
            )
        else:
            preview_exprs.append(pl.col(column_name))
    return df.select(preview_exprs)


def stage8_preview_table(product: dict, n_rows: int = STAGE8_TABLE_PREVIEW_ROWS) -> pl.DataFrame:
    path = Path(product["path"])
    fmt = product.get("format")
    if fmt == "parquet":
        preview = pl.scan_parquet(path).head(n_rows).collect()
    elif fmt == "csv":
        preview = pl.read_csv(path, n_rows=n_rows, infer_schema_length=max(n_rows, 100))
    else:
        raise ValueError(f"Unsupported table format for preview: {fmt}")
    return stage8_compact_preview(preview)

stage8_table_products = [product for product in stage8_products if product.get("format") in {"parquet", "csv"}]

display(Markdown(
    f"### Stage 8 Table Artifact Previews\n\n"
    f"Showing the first `{STAGE8_TABLE_PREVIEW_ROWS}` rows of each CSV/parquet artifact. "
    "Large customer and embedding files are read lazily, so this is a sanity preview rather than a full-table load."
))

for product in stage8_table_products:
    row_count = product.get("row_count")
    row_count_text = "unknown" if row_count is None else f"{int(row_count):,}"
    schema = product.get("schema") or []
    display(Markdown(
        f"#### `{product['name']}`\n\n"
        f"File: `{Path(product['path']).name}`  \n"
        f"Rows: `{row_count_text}` | Columns: `{len(schema)}` | Format: `{product.get('format')}`"
    ))
    display(stage8_preview_table(product))

if not stage8_non_table_artifacts.is_empty():
    display(Markdown("### Non-Table Stage 8 Artifacts"))
    display(stage8_non_table_artifacts.select(["name", "format", "exists", "size_mb", "path"]))
